# Validação de séries temporais. Barragem de Alqueva

## Representação de séries dV e dH de asc LOS, desc LOS, asc/desc IDW, ortho

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]
desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]
ortho_v = ortho_v[(ortho_v['northing'] >= norte_min) & (ortho_v['northing'] <= norte_max) &
                  (ortho_v['easting'] >= este_min) & (ortho_v['easting'] <= este_max)]
ortho_h = ortho_h[(ortho_h['northing'] >= norte_min) & (ortho_h['northing'] <= norte_max) &
                  (ortho_h['easting'] >= este_min) & (ortho_h['easting'] <= este_max)]

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # Ajustar conforme as colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

agg = asc_interp.dropna(subset=['cell_x','cell_y']).groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. GeoDataFrames
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)
points = agg.groupby('cell_id').first().reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['x_center'], points['y_center']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 10. Usar TODAS as células com dados
# ==============================
selected_ids = agg['cell_id'].unique().tolist()
agg_sel = agg.copy()
grid_sel = grid[grid['cell_id'].isin(selected_ids)]
points_sel = gdf_points[gdf_points['cell_id'].isin(selected_ids)]



# ==============================
# 11. ORTHO em formato longo e atribuição de células
# ==============================
def melt_ortho(df):
    disp_cols = df.columns[24:]  # ajustar conforme datas ORTHO
    long_df = df.melt(id_vars=['easting','northing'], value_vars=disp_cols,
                      var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

# ==============================
# Transformar ORTHO em formato longo
# ==============================
ortho_v_long = melt_ortho(ortho_v)
ortho_h_long = melt_ortho(ortho_h)

ortho_v_sel = ortho_v_long.copy()
ortho_h_sel = ortho_h_long.copy()


# atribuir células
for ortho_df in [ortho_v_long, ortho_h_long]:
    ortho_df['cell_x'] = pd.cut(ortho_df['easting'], bins=x_edges_shifted, labels=False)
    ortho_df['cell_y'] = pd.cut(ortho_df['northing'], bins=y_edges_shifted, labels=False)
    ortho_df.dropna(subset=['cell_x','cell_y'], inplace=True)
    ortho_df['cell_id'] = ortho_df['cell_x'].astype(int).astype(str) + "_" + ortho_df['cell_y'].astype(int).astype(str)

ortho_v_sel = ortho_v_long[ortho_v_long['cell_id'].isin(selected_ids)]
ortho_h_sel = ortho_h_long[ortho_h_long['cell_id'].isin(selected_ids)]

# ==============================
# 12. Mapa das células selecionadas + pontos ORTHO (com legenda organizada)
# ==============================

# Criar figura
fig_map, ax_map = plt.subplots(figsize=(10, 10))

# Grelha base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5, 
                   label=f'Grelha Base ({grid_size} m)')

# Células selecionadas
grid_sel.boundary.plot(ax=ax_map, color='red', linewidth=1.5, 
                       label=f'Células Selecionadas ({grid_size} m)')

# Pontos ASC/DESC (centros das células)
points_sel.plot(ax=ax_map, color='white', edgecolor='black', markersize=60, label='Pontos ASC/DESC')

# Criar GeoDataFrame combinado dos pontos ORTHO selecionados
ortho_comb = pd.concat([ortho_v_sel, ortho_h_sel], ignore_index=True)

# Evitar duplicados exatos
ortho_comb = ortho_comb.drop_duplicates(subset=['easting', 'northing', 'date'])

# Transformar em GeoDataFrame
gdf_ortho_comb = gpd.GeoDataFrame(
    ortho_comb,
    geometry=gpd.points_from_xy(ortho_comb['easting'], ortho_comb['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)


# Pontos ORTHO combinados (preto)
if not gdf_ortho_comb.empty:
    gdf_ortho_comb.plot(ax=ax_map, color='black', markersize=25, alpha=0.8, label='Pontos ORTHO')

# IDs das células
for _, row in grid_sel.iterrows():
    x_min, y_min, x_max, y_max = row['geometry'].bounds
    ax_map.text(
        x_min, y_max, row['cell_id'],
        fontsize=9, ha='left', va='top', color='black',
        bbox=dict(facecolor='white', alpha=0.6, pad=1)
    )

# Adicionar basemap e formato
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title("Mapa das Células Selecionadas com Pontos ASC/DESC e ORTHO", fontsize=14)
ax_map.set_axis_off()

# Legenda organizada (inclui o tamanho da grelha)
ax_map.legend(loc='lower left', fontsize=9, frameon=True)

plt.show()


# ==============================
# 13. Grid de gráficos - TODAS AS CÉLULAS
# ==============================

# Usar todas as células disponíveis dentro dos limites
agg_all = agg.copy()

# Calcular limites globais (min e max de todos os dados)
v_min = min(
    agg_all['dV'].min(),
    agg_all['dH'].min(),
    asc_long['disp'].min() if not asc_long.empty else np.inf,
    desc_long['disp'].min() if not desc_long.empty else np.inf,
    ortho_v_long['disp'].min() if not ortho_v_long.empty else np.inf,
    ortho_h_long['disp'].min() if not ortho_h_long.empty else np.inf
)
v_max = max(
    agg_all['dV'].max(),
    agg_all['dH'].max(),
    asc_long['disp'].max() if not asc_long.empty else -np.inf,
    desc_long['disp'].max() if not desc_long.empty else -np.inf,
    ortho_v_long['disp'].max() if not ortho_v_long.empty else -np.inf,
    ortho_h_long['disp'].max() if not ortho_h_long.empty else -np.inf
)

# Todas as células (ordenadas por Y e X)
all_ids = sorted(agg_all['cell_id'].unique(), key=lambda s: (int(s.split('_')[1]), int(s.split('_')[0])))

# Número de células e configuração da figura
n_cells = len(all_ids)
n_cols = 3  # reduzido para 3 colunas
n_rows = int(np.ceil(n_cells / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows*4), sharex=True, sharey=True)
if n_rows == 1:
    axes = axes.reshape(1, -1)

for i, cid in enumerate(all_ids):
    r = i // n_cols
    c = i % n_cols
    ax = axes[r, c]

    agg_cell = agg_all[agg_all['cell_id'] == cid]
    ortho_v_cell = ortho_v_long[ortho_v_long['cell_id'] == cid]
    ortho_h_cell = ortho_h_long[ortho_h_long['cell_id'] == cid]

    # ASC/DESC combinados (IDW)
    ax.plot(agg_cell['date'], agg_cell['dV'], color='blue', label='ASC/DESC dV IDW')
    ax.plot(agg_cell['date'], agg_cell['dH'], color='green', label='ASC/DESC dH IDW')

    # ASC LOS
    asc_cell = asc_long.copy()
    asc_cell['cell_x'] = pd.cut(asc_cell['easting'], bins=x_edges_shifted, labels=False)
    asc_cell['cell_y'] = pd.cut(asc_cell['northing'], bins=y_edges_shifted, labels=False)
    asc_cell.dropna(subset=['cell_x','cell_y'], inplace=True)
    asc_cell['cell_id'] = asc_cell['cell_x'].astype(int).astype(str) + "_" + asc_cell['cell_y'].astype(int).astype(str)
    asc_cell = asc_cell[asc_cell['cell_id'] == cid]
    if not asc_cell.empty:
        ax.plot(asc_cell['date'], asc_cell['disp'], color='cyan', linestyle='-', label='ASC LOS')

    # DESC LOS
    desc_cell = desc_long.copy()
    desc_cell['cell_x'] = pd.cut(desc_cell['easting'], bins=x_edges_shifted, labels=False)
    desc_cell['cell_y'] = pd.cut(desc_cell['northing'], bins=y_edges_shifted, labels=False)
    desc_cell.dropna(subset=['cell_x','cell_y'], inplace=True)
    desc_cell['cell_id'] = desc_cell['cell_x'].astype(int).astype(str) + "_" + desc_cell['cell_y'].astype(int).astype(str)
    desc_cell = desc_cell[desc_cell['cell_id'] == cid]
    if not desc_cell.empty:
        ax.plot(desc_cell['date'], desc_cell['disp'], color='magenta', linestyle='-', label='DESC LOS')

    # ORTHO (tracejado)
    if not ortho_v_cell.empty:
        ax.plot(ortho_v_cell['date'], ortho_v_cell['disp'], color='blue', linestyle='--', label='ORTHO dV')
    if not ortho_h_cell.empty:
        ax.plot(ortho_h_cell['date'], ortho_h_cell['disp'], color='green', linestyle='--', label='ORTHO dH')

    ax.set_title(f"Célula {cid}", fontsize=9)
    ax.set_ylabel("Deslocamento (mm)")
    ax.set_xlabel("Data")
    ax.set_ylim(v_min, v_max)  # mantém mesma escala para todos
    ax.legend(fontsize=7, loc='best')

# Remover eixos vazios
for j in range(i+1, n_rows*n_cols):
    r = j // n_cols
    c = j % n_cols
    fig.delaxes(axes[r, c])

plt.tight_layout()
plt.show()


## Representação de séries dV e dH de asc/desc IDW, ortho 

## Todas as células

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]
desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]
ortho_v = ortho_v[(ortho_v['northing'] >= norte_min) & (ortho_v['northing'] <= norte_max) &
                  (ortho_v['easting'] >= este_min) & (ortho_v['easting'] <= este_max)]
ortho_h = ortho_h[(ortho_h['northing'] >= norte_min) & (ortho_h['northing'] <= norte_max) &
                  (ortho_h['easting'] >= este_min) & (ortho_h['easting'] <= este_max)]

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # Ajustar conforme as colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

agg = asc_interp.dropna(subset=['cell_x','cell_y']).groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. GeoDataFrames
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)
points = agg.groupby('cell_id').first().reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['x_center'], points['y_center']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 10. Usar TODAS as células com dados
# ==============================
selected_ids = agg['cell_id'].unique().tolist()
agg_sel = agg.copy()
grid_sel = grid[grid['cell_id'].isin(selected_ids)]
points_sel = gdf_points[gdf_points['cell_id'].isin(selected_ids)]



# ==============================
# 11. ORTHO em formato longo e atribuição de células
# ==============================
def melt_ortho(df):
    disp_cols = df.columns[24:]  # ajustar conforme datas ORTHO
    long_df = df.melt(id_vars=['easting','northing'], value_vars=disp_cols,
                      var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

# ==============================
# Transformar ORTHO em formato longo
# ==============================
ortho_v_long = melt_ortho(ortho_v)
ortho_h_long = melt_ortho(ortho_h)

ortho_v_sel = ortho_v_long.copy()
ortho_h_sel = ortho_h_long.copy()


# atribuir células
for ortho_df in [ortho_v_long, ortho_h_long]:
    ortho_df['cell_x'] = pd.cut(ortho_df['easting'], bins=x_edges_shifted, labels=False)
    ortho_df['cell_y'] = pd.cut(ortho_df['northing'], bins=y_edges_shifted, labels=False)
    ortho_df.dropna(subset=['cell_x','cell_y'], inplace=True)
    ortho_df['cell_id'] = ortho_df['cell_x'].astype(int).astype(str) + "_" + ortho_df['cell_y'].astype(int).astype(str)

ortho_v_sel = ortho_v_long[ortho_v_long['cell_id'].isin(selected_ids)]
ortho_h_sel = ortho_h_long[ortho_h_long['cell_id'].isin(selected_ids)]

# ==============================
# 12. Mapa das células selecionadas + pontos ORTHO (com legenda organizada)
# ==============================

# Criar figura
fig_map, ax_map = plt.subplots(figsize=(10, 10))

# Grelha base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5, 
                   label=f'Grelha Base ({grid_size} m)')

# Células selecionadas
grid_sel.boundary.plot(ax=ax_map, color='red', linewidth=1.5, 
                       label=f'Células Selecionadas ({grid_size} m)')

# Pontos ASC/DESC (centros das células)
points_sel.plot(ax=ax_map, color='white', edgecolor='black', markersize=60, label='Pontos ASC/DESC')

# Criar GeoDataFrame combinado dos pontos ORTHO selecionados
ortho_comb = pd.concat([ortho_v_sel, ortho_h_sel], ignore_index=True)

# Evitar duplicados exatos
ortho_comb = ortho_comb.drop_duplicates(subset=['easting', 'northing', 'date'])

# Transformar em GeoDataFrame
gdf_ortho_comb = gpd.GeoDataFrame(
    ortho_comb,
    geometry=gpd.points_from_xy(ortho_comb['easting'], ortho_comb['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)


# Pontos ORTHO combinados (preto)
if not gdf_ortho_comb.empty:
    gdf_ortho_comb.plot(ax=ax_map, color='black', markersize=25, alpha=0.8, label='Pontos ORTHO')

# IDs das células
for _, row in grid_sel.iterrows():
    x_min, y_min, x_max, y_max = row['geometry'].bounds
    ax_map.text(
        x_min, y_max, row['cell_id'],
        fontsize=9, ha='left', va='top', color='black',
        bbox=dict(facecolor='white', alpha=0.6, pad=1)
    )

# Adicionar basemap e formato
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title("Mapa das Células Selecionadas com Pontos ASC/DESC e ORTHO", fontsize=14)
ax_map.set_axis_off()

# Legenda organizada (inclui o tamanho da grelha)
ax_map.legend(loc='lower left', fontsize=9, frameon=True)

plt.show()


# ==============================
# 13. Grid de gráficos - TODAS AS CÉLULAS (3 colunas)
# ==============================

# Usar todas as células disponíveis dentro dos limites
agg_all = agg.copy()

# Calcular limites globais (min e max de todos os dados)
v_min = min(
    agg_all['dV'].min(),
    agg_all['dH'].min(),
    ortho_v_long['disp'].min() if not ortho_v_long.empty else np.inf,
    ortho_h_long['disp'].min() if not ortho_h_long.empty else np.inf
)
v_max = max(
    agg_all['dV'].max(),
    agg_all['dH'].max(),
    ortho_v_long['disp'].max() if not ortho_v_long.empty else -np.inf,
    ortho_h_long['disp'].max() if not ortho_h_long.empty else -np.inf
)

# Todas as células (ordenadas por Y e X)
all_ids = sorted(agg_all['cell_id'].unique(), key=lambda s: (int(s.split('_')[1]), int(s.split('_')[0])))

# Número de células
n_cells = len(all_ids)
n_cols = 3  # reduzido para 3 colunas
n_rows = int(np.ceil(n_cells / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows*5), sharex=True, sharey=True)  # ajustei a altura

if n_rows == 1:
    axes = axes.reshape(1, -1)

for i, cid in enumerate(all_ids):
    r = i // n_cols
    c = i % n_cols
    ax = axes[r, c]

    agg_cell = agg_all[agg_all['cell_id'] == cid]
    ortho_v_cell = ortho_v_long[ortho_v_long['cell_id'] == cid]
    ortho_h_cell = ortho_h_long[ortho_h_long['cell_id'] == cid]

    # ASC/DESC
    ax.plot(agg_cell['date'], agg_cell['dV'], color='blue', label='ASC/DESC dV')
    ax.plot(agg_cell['date'], agg_cell['dH'], color='green', label='ASC/DESC dH')

    # ORTHO (tracejado)
    if not ortho_v_cell.empty:
        ax.plot(ortho_v_cell['date'], ortho_v_cell['disp'], color='blue', linestyle='--', label='ORTHO dV')
    if not ortho_h_cell.empty:
        ax.plot(ortho_h_cell['date'], ortho_h_cell['disp'], color='green', linestyle='--', label='ORTHO dH')

    ax.set_title(f"Célula {cid}", fontsize=9)
    ax.set_ylabel("Deslocamento (mm)")
    ax.set_xlabel("Data")
    ax.set_ylim(v_min, v_max)
    ax.legend(fontsize=7, loc='best')

# Remove eixos vazios
for j in range(i+1, n_rows*n_cols):
    r = j // n_cols
    c = j % n_cols
    fig.delaxes(axes[r, c])

plt.tight_layout()
plt.show()


## Pearson, r

In [ ]:
import seaborn as sns
from scipy.stats import pearsonr

# ==============================
# 14. Correlação InSAR vs ORTHO por célula
# ==============================

import seaborn as sns

# Lista de células (mesma ordem que antes)
all_ids = sorted(agg_all['cell_id'].unique(), key=lambda s: (int(s.split('_')[1]), int(s.split('_')[0])))

n_cells = len(all_ids)
n_cols = 3
n_rows = int(np.ceil(n_cells / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows*5), sharex=True, sharey=True)

if n_rows == 1:
    axes = axes.reshape(1, -1)

for i, cid in enumerate(all_ids):
    r = i // n_cols
    c = i % n_cols
    ax = axes[r, c]

    # Extrair dados da célula
    agg_cell = agg_all[agg_all['cell_id'] == cid]
    ortho_v_cell = ortho_v_long[ortho_v_long['cell_id'] == cid]
    ortho_h_cell = ortho_h_long[ortho_h_long['cell_id'] == cid]

    # Combinar séries (InSAR vs ORTHO)
    merged_v = pd.merge(
        agg_cell[['date', 'dV']], 
        ortho_v_cell[['date', 'disp']], 
        on='date', 
        how='inner'
    ).rename(columns={'dV': 'disp_insar', 'disp': 'disp_ortho'})

    merged_h = pd.merge(
        agg_cell[['date', 'dH']], 
        ortho_h_cell[['date', 'disp']], 
        on='date', 
        how='inner'
    ).rename(columns={'dH': 'disp_insar', 'disp': 'disp_ortho'})

    # Calcular correlação
    r_v = merged_v['disp_insar'].corr(merged_v['disp_ortho']) if len(merged_v) > 1 else np.nan
    r_h = merged_h['disp_insar'].corr(merged_h['disp_ortho']) if len(merged_h) > 1 else np.nan

    # Scatter plot
    if not merged_v.empty:
        ax.scatter(merged_v['disp_ortho'], merged_v['disp_insar'], color='blue', alpha=0.6, label=f'dV (r={r_v:.2f})')
    if not merged_h.empty:
        ax.scatter(merged_h['disp_ortho'], merged_h['disp_insar'], color='green', alpha=0.6, label=f'dH (r={r_h:.2f})')

    # Reta identidade
    if not merged_v.empty or not merged_h.empty:
        vmin = np.nanmin([
            merged_v['disp_ortho'].min() if not merged_v.empty else np.nan,
            merged_h['disp_ortho'].min() if not merged_h.empty else np.nan
        ])
        vmax = np.nanmax([
            merged_v['disp_ortho'].max() if not merged_v.empty else np.nan,
            merged_h['disp_ortho'].max() if not merged_h.empty else np.nan
        ])
        ax.plot([vmin, vmax], [vmin, vmax], 'k--', lw=1)

    ax.set_title(f"Célula {cid}", fontsize=9)
    ax.set_xlabel("ORTHO (mm)")
    ax.set_ylabel("InSAR (mm)")
    ax.legend(fontsize=7, loc='best')
    ax.grid(False)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)


# Apagar eixos vazios
for j in range(i+1, n_rows*n_cols):
    r = j // n_cols
    c = j % n_cols
    fig.delaxes(axes[r, c])

plt.tight_layout()
plt.show()


## Distribuição do erro por célula no mapa

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import contextily as ctx
from mpl_toolkits.axes_grid1 import make_axes_locatable
import geopandas as gpd
import pandas as pd
import numpy as np
import geopandas as gpd

# Criar lista para armazenar erros por célula
error_list = []

for cid in agg_all['cell_id'].unique():
    agg_cell = agg_all[agg_all['cell_id'] == cid].iloc[0]  # único ponto ASC/DESC
    ortho_v_cell = ortho_v_long[ortho_v_long['cell_id'] == cid]
    ortho_h_cell = ortho_h_long[ortho_h_long['cell_id'] == cid]

    # Se não houver ORTHO, ignorar
    if ortho_v_cell.empty or ortho_h_cell.empty:
        continue

    # Calcular erro absoluto em relação ao ORTHO
    err_dV = np.abs(agg_cell['dV'] - ortho_v_cell['disp'].mean())
    err_dH = np.abs(agg_cell['dH'] - ortho_h_cell['disp'].mean())

    error_list.append({'cell_id': cid, 'err_dV': err_dV, 'err_dH': err_dH})

# Transformar em DataFrame
error_df = pd.DataFrame(error_list)

# Juntar com a grelha
grid_error = grid.merge(error_df, on='cell_id', how='left')
grid_error = gpd.GeoDataFrame(grid_error, geometry='geometry', crs="EPSG:3857")


# ==============================
# Escala comum para cores
# ==============================
vmin = 0
vmax = max(grid_error['err_dV'].max(), grid_error['err_dH'].max())

cmap_dV = mpl.cm.Oranges
cmap_dH = mpl.cm.Greens
norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

# ==============================
# Criar figura com 2 mapas
# ==============================
fig, axes = plt.subplots(1, 2, figsize=(20, 12))

# -------- dV ------------
im = grid_error.plot(
    column='err_dV', ax=axes[0],
    cmap=cmap_dV, edgecolor='black',
    linewidth=0.5, alpha=0.4
)
ctx.add_basemap(axes[0], source=ctx.providers.Esri.WorldImagery)
axes[0].set_axis_off()
axes[0].set_title('Erro dV ASC/DESC vs ORTHO', fontsize=16)

divider = make_axes_locatable(axes[0])
cax = divider.append_axes("right", size="3%", pad=0.05)
sm = mpl.cm.ScalarMappable(cmap=cmap_dV, norm=norm)
sm._A = []
fig.colorbar(sm, cax=cax, label="Erro (mm)")

# -------- dH ------------
im = grid_error.plot(
    column='err_dH', ax=axes[1],
    cmap=cmap_dH, edgecolor='black',
    linewidth=0.5, alpha=0.4
)
ctx.add_basemap(axes[1], source=ctx.providers.Esri.WorldImagery)
axes[1].set_axis_off()
axes[1].set_title('Erro dH ASC/DESC vs ORTHO', fontsize=16)

divider = make_axes_locatable(axes[1])
cax = divider.append_axes("right", size="3%", pad=0.05)
sm = mpl.cm.ScalarMappable(cmap=cmap_dH, norm=norm)
sm._A = []
fig.colorbar(sm, cax=cax, label="Erro (mm)")

plt.tight_layout()
plt.show()


## RMSE e MAE para asc/desc em comparação ccom ortho

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =====================================
# 1. Calcular erro ASC/DESC vs ORTHO por célula e data
# =====================================

def compute_error(agg_df, ortho_v, ortho_h):
    # VERTICAL
    df_v = pd.merge(
        agg_df[['cell_id','date','dV']],
        ortho_v[['cell_id','date','disp']],
        on=['cell_id','date'], how='inner'
    ).rename(columns={'dV':'ascdesc','disp':'ortho'})
    df_v['err'] = df_v['ascdesc'] - df_v['ortho']

    # HORIZONTAL
    df_h = pd.merge(
        agg_df[['cell_id','date','dH']],
        ortho_h[['cell_id','date','disp']],
        on=['cell_id','date'], how='inner'
    ).rename(columns={'dH':'ascdesc','disp':'ortho'})
    df_h['err'] = df_h['ascdesc'] - df_h['ortho']

    df_v['Componente'] = 'dV'
    df_h['Componente'] = 'dH'

    df_err = pd.concat([df_v, df_h], ignore_index=True)
    return df_err[['cell_id','date','Componente','ascdesc','ortho','err']]

df_error = compute_error(agg_all, ortho_v_long, ortho_h_long)

# =====================================
# 2. Cálculo das métricas globais (RMSE e MAE)
# =====================================

metrics = []

for comp in ['dV', 'dH']:
    df = df_error[df_error['Componente'] == comp]
    err = df['err'].values

    rmse = np.sqrt(np.mean(err**2))
    mae = np.mean(np.abs(err))

    metrics.append({'Componente': comp, 'RMSE': rmse, 'MAE': mae})

metrics_df = pd.DataFrame(metrics)
print(metrics_df)

# =====================================
# 3. Gráfico de barras (RMSE e MAE)
# =====================================

x = np.arange(len(metrics_df['Componente']))
width = 0.3

fig, ax = plt.subplots(figsize=(7,5))

ax.bar(x - width/2, metrics_df['RMSE'], width, label='RMSE', color='#7f7f7f')   # cinza
ax.bar(x + width/2, metrics_df['MAE'], width, label='MAE', color='#1f77b4')     # azul

ax.set_xticks(x)
ax.set_xticklabels(metrics_df['Componente'], fontsize=12)
ax.set_ylabel('Erro (mm)', fontsize=12)
ax.set_title('Erro Global ASC/DESC vs ORTHO (RMSE e MAE)', fontsize=14)
ax.legend(frameon=False)
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


## Distribuição do erro por célula em plot

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =====================================
# 1️⃣ Calcular erro ASC/DESC vs ORTHO por célula (com sinal)
# =====================================

error_list = []

for cid in agg_all['cell_id'].unique():
    agg_cell = agg_all[agg_all['cell_id'] == cid].iloc[0]  # único ponto ASC/DESC
    ortho_v_cell = ortho_v_long[ortho_v_long['cell_id'] == cid]
    ortho_h_cell = ortho_h_long[ortho_h_long['cell_id'] == cid]

    # Se não houver ORTHO, ignora
    if ortho_v_cell.empty or ortho_h_cell.empty:
        continue

    # Erro com sinal (sem np.abs)
    err_dV = agg_cell['dV'] - ortho_v_cell['disp'].mean()
    err_dH = agg_cell['dH'] - ortho_h_cell['disp'].mean()

    error_list.append({'cell_id': cid, 'err_dV': err_dV, 'err_dH': err_dH})

# Criar DataFrame de erros
df_compare = pd.DataFrame(error_list)

# =====================================
# 2️⃣ Transformar para formato longo para o Seaborn
# =====================================
df_long = df_compare.melt(
    id_vars='cell_id',
    value_vars=['err_dV', 'err_dH'],
    var_name='Componente',
    value_name='Erro'
)

# =====================================
# 3️⃣ Gráfico com erro com sinal
# =====================================
plt.figure(figsize=(14,6))
sns.barplot(
    x='cell_id', 
    y='Erro', 
    hue='Componente', 
    data=df_long, 
    palette=['#d62728','#1f77b4'],  # vermelho e azul sóbrios
    errorbar=None
)

plt.axhline(0, color='black', linewidth=1)
plt.xticks(rotation=90)
plt.ylabel('Erro (mm)')
plt.title('Erro ASC/DESC vs ORTHO por célula (com sinal)')
plt.legend(title='Componente')
plt.tight_layout()
plt.show()


## Céluas selecionadas

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]
desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]
ortho_v = ortho_v[(ortho_v['northing'] >= norte_min) & (ortho_v['northing'] <= norte_max) &
                  (ortho_v['easting'] >= este_min) & (ortho_v['easting'] <= este_max)]
ortho_h = ortho_h[(ortho_h['northing'] >= norte_min) & (ortho_h['northing'] <= norte_max) &
                  (ortho_h['easting'] >= este_min) & (ortho_h['easting'] <= este_max)]

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # Ajustar conforme as colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

agg = asc_interp.dropna(subset=['cell_x','cell_y']).groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. GeoDataFrames
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)
points = agg.groupby('cell_id').first().reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['x_center'], points['y_center']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 10. Usar TODAS as células com dados
# ==============================
selected_ids = agg['cell_id'].unique().tolist()
agg_sel = agg.copy()
grid_sel = grid[grid['cell_id'].isin(selected_ids)]
points_sel = gdf_points[gdf_points['cell_id'].isin(selected_ids)]



# ==============================
# 11. ORTHO em formato longo e atribuição de células
# ==============================
def melt_ortho(df):
    disp_cols = df.columns[24:]  # ajustar conforme datas ORTHO
    long_df = df.melt(id_vars=['easting','northing'], value_vars=disp_cols,
                      var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

# ==============================
# Transformar ORTHO em formato longo
# ==============================
ortho_v_long = melt_ortho(ortho_v)
ortho_h_long = melt_ortho(ortho_h)

ortho_v_sel = ortho_v_long.copy()
ortho_h_sel = ortho_h_long.copy()


# atribuir células
for ortho_df in [ortho_v_long, ortho_h_long]:
    ortho_df['cell_x'] = pd.cut(ortho_df['easting'], bins=x_edges_shifted, labels=False)
    ortho_df['cell_y'] = pd.cut(ortho_df['northing'], bins=y_edges_shifted, labels=False)
    ortho_df.dropna(subset=['cell_x','cell_y'], inplace=True)
    ortho_df['cell_id'] = ortho_df['cell_x'].astype(int).astype(str) + "_" + ortho_df['cell_y'].astype(int).astype(str)

ortho_v_sel = ortho_v_long[ortho_v_long['cell_id'].isin(selected_ids)]
ortho_h_sel = ortho_h_long[ortho_h_long['cell_id'].isin(selected_ids)]

# ==============================
# 12. Mapa das células selecionadas + pontos ORTHO (com legenda organizada)
# ==============================

# Criar figura
fig_map, ax_map = plt.subplots(figsize=(10, 10))

# Grelha base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5, 
                   label=f'Grelha Base ({grid_size} m)')

# Células selecionadas
grid_sel.boundary.plot(ax=ax_map, color='red', linewidth=1.5, 
                       label=f'Células Selecionadas ({grid_size} m)')

# Pontos ASC/DESC (centros das células)
points_sel.plot(ax=ax_map, color='white', edgecolor='black', markersize=60, label='Pontos ASC/DESC')

# Criar GeoDataFrame combinado dos pontos ORTHO selecionados
ortho_comb = pd.concat([ortho_v_sel, ortho_h_sel], ignore_index=True)

# Evitar duplicados exatos
ortho_comb = ortho_comb.drop_duplicates(subset=['easting', 'northing', 'date'])

# Transformar em GeoDataFrame
gdf_ortho_comb = gpd.GeoDataFrame(
    ortho_comb,
    geometry=gpd.points_from_xy(ortho_comb['easting'], ortho_comb['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)


# Pontos ORTHO combinados (preto)
if not gdf_ortho_comb.empty:
    gdf_ortho_comb.plot(ax=ax_map, color='black', markersize=25, alpha=0.8, label='Pontos ORTHO')

# IDs das células
for _, row in grid_sel.iterrows():
    x_min, y_min, x_max, y_max = row['geometry'].bounds
    ax_map.text(
        x_min, y_max, row['cell_id'],
        fontsize=9, ha='left', va='top', color='black',
        bbox=dict(facecolor='white', alpha=0.6, pad=1)
    )

# Adicionar basemap e formato
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title("Mapa das Células Selecionadas com Pontos ASC/DESC e ORTHO", fontsize=14)
ax_map.set_axis_off()

# Legenda organizada (inclui o tamanho da grelha)
ax_map.legend(loc='lower left', fontsize=9, frameon=True)

plt.show()


# ==============================
# 13. Grid de gráficos - CÉLULAS SELECIONADAS COM LOS
# ==============================

# Lista de células escolhidas
selected_ids = ["3_5","4_5","5_5","6_5","7_5",
                "3_4","4_4","5_4","6_4",
                "5_6","6_6","7_6"]

# Selecionar dados das células
agg_sel_cells = agg[agg['cell_id'].isin(selected_ids)]

# Calcular limites globais (mesma escala)
v_min = min(
    agg_sel_cells['dV'].min(),
    agg_sel_cells['dH'].min(),
    asc_long['disp'].min(),
    desc_long['disp'].min(),
    ortho_v_sel['disp'].min(),
    ortho_h_sel['disp'].min()
)
v_max = max(
    agg_sel_cells['dV'].max(),
    agg_sel_cells['dH'].max(),
    asc_long['disp'].max(),
    desc_long['disp'].max(),
    ortho_v_sel['disp'].max(),
    ortho_h_sel['disp'].max()
)

# Configuração do grid
n_cells = len(selected_ids)
n_cols = 3
n_rows = int(np.ceil(n_cells / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows*4), sharex=True, sharey=True)
if n_rows == 1:
    axes = axes.reshape(1, -1)

for i, cid in enumerate(selected_ids):
    r = i // n_cols
    c = i % n_cols
    ax = axes[r, c]

    # Dados da célula
    agg_cell = agg_sel_cells[agg_sel_cells['cell_id'] == cid]
    ortho_v_cell = ortho_v_sel[ortho_v_sel['cell_id'] == cid]
    ortho_h_cell = ortho_h_sel[ortho_h_sel['cell_id'] == cid]

    # ASC LOS
    asc_cell = asc_long.copy()
    asc_cell['cell_x'] = pd.cut(asc_cell['easting'], bins=x_edges_shifted, labels=False)
    asc_cell['cell_y'] = pd.cut(asc_cell['northing'], bins=y_edges_shifted, labels=False)
    asc_cell.dropna(subset=['cell_x','cell_y'], inplace=True)
    asc_cell['cell_id'] = asc_cell['cell_x'].astype(int).astype(str) + "_" + asc_cell['cell_y'].astype(int).astype(str)
    asc_cell = asc_cell[asc_cell['cell_id'] == cid]

    # DESC LOS
    desc_cell = desc_long.copy()
    desc_cell['cell_x'] = pd.cut(desc_cell['easting'], bins=x_edges_shifted, labels=False)
    desc_cell['cell_y'] = pd.cut(desc_cell['northing'], bins=y_edges_shifted, labels=False)
    desc_cell.dropna(subset=['cell_x','cell_y'], inplace=True)
    desc_cell['cell_id'] = desc_cell['cell_x'].astype(int).astype(str) + "_" + desc_cell['cell_y'].astype(int).astype(str)
    desc_cell = desc_cell[desc_cell['cell_id'] == cid]

    # Plots
    if not agg_cell.empty:
        ax.plot(agg_cell['date'], agg_cell['dV'], color='blue', label='ASC/DESC dV IDW')
        ax.plot(agg_cell['date'], agg_cell['dH'], color='green', label='ASC/DESC dH IDW')
    if not asc_cell.empty:
        ax.plot(asc_cell['date'], asc_cell['disp'], color='cyan', linestyle='-', label='ASC LOS')
    if not desc_cell.empty:
        ax.plot(desc_cell['date'], desc_cell['disp'], color='magenta', linestyle='-', label='DESC LOS')
    if not ortho_v_cell.empty:
        ax.plot(ortho_v_cell['date'], ortho_v_cell['disp'], color='blue', linestyle='--', label='ORTHO dV')
    if not ortho_h_cell.empty:
        ax.plot(ortho_h_cell['date'], ortho_h_cell['disp'], color='green', linestyle='--', label='ORTHO dH')

    ax.set_title(f"Célula {cid}", fontsize=10)
    ax.set_ylabel("Deslocamento (mm)")
    ax.set_xlabel("Data")
    ax.set_ylim(v_min, v_max)
    ax.legend(fontsize=8, loc='best')

# Remover eixos vazios
for j in range(i+1, n_rows*n_cols):
    r = j // n_cols
    c = j % n_cols
    fig.delaxes(axes[r, c])

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]
desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]
ortho_v = ortho_v[(ortho_v['northing'] >= norte_min) & (ortho_v['northing'] <= norte_max) &
                  (ortho_v['easting'] >= este_min) & (ortho_v['easting'] <= este_max)]
ortho_h = ortho_h[(ortho_h['northing'] >= norte_min) & (ortho_h['northing'] <= norte_max) &
                  (ortho_h['easting'] >= este_min) & (ortho_h['easting'] <= este_max)]

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # Ajustar conforme as colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

agg = asc_interp.dropna(subset=['cell_x','cell_y']).groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. GeoDataFrames
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)
points = agg.groupby('cell_id').first().reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['x_center'], points['y_center']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 10. Usar TODAS as células com dados
# ==============================
selected_ids = agg['cell_id'].unique().tolist()
agg_sel = agg.copy()
grid_sel = grid[grid['cell_id'].isin(selected_ids)]
points_sel = gdf_points[gdf_points['cell_id'].isin(selected_ids)]



# ==============================
# 11. ORTHO em formato longo e atribuição de células
# ==============================
def melt_ortho(df):
    disp_cols = df.columns[24:]  # ajustar conforme datas ORTHO
    long_df = df.melt(id_vars=['easting','northing'], value_vars=disp_cols,
                      var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

# ==============================
# Transformar ORTHO em formato longo
# ==============================
ortho_v_long = melt_ortho(ortho_v)
ortho_h_long = melt_ortho(ortho_h)

ortho_v_sel = ortho_v_long.copy()
ortho_h_sel = ortho_h_long.copy()


# atribuir células
for ortho_df in [ortho_v_long, ortho_h_long]:
    ortho_df['cell_x'] = pd.cut(ortho_df['easting'], bins=x_edges_shifted, labels=False)
    ortho_df['cell_y'] = pd.cut(ortho_df['northing'], bins=y_edges_shifted, labels=False)
    ortho_df.dropna(subset=['cell_x','cell_y'], inplace=True)
    ortho_df['cell_id'] = ortho_df['cell_x'].astype(int).astype(str) + "_" + ortho_df['cell_y'].astype(int).astype(str)

ortho_v_sel = ortho_v_long[ortho_v_long['cell_id'].isin(selected_ids)]
ortho_h_sel = ortho_h_long[ortho_h_long['cell_id'].isin(selected_ids)]

# ==============================
# 12. Mapa das células selecionadas + pontos ORTHO (com legenda organizada)
# ==============================

# Criar figura
fig_map, ax_map = plt.subplots(figsize=(10, 10))

# Grelha base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5, 
                   label=f'Grelha Base ({grid_size} m)')

# Células selecionadas
grid_sel.boundary.plot(ax=ax_map, color='red', linewidth=1.5, 
                       label=f'Células Selecionadas ({grid_size} m)')

# Pontos ASC/DESC (centros das células)
points_sel.plot(ax=ax_map, color='white', edgecolor='black', markersize=60, label='Pontos ASC/DESC')

# Criar GeoDataFrame combinado dos pontos ORTHO selecionados
ortho_comb = pd.concat([ortho_v_sel, ortho_h_sel], ignore_index=True)

# Evitar duplicados exatos
ortho_comb = ortho_comb.drop_duplicates(subset=['easting', 'northing', 'date'])

# Transformar em GeoDataFrame
gdf_ortho_comb = gpd.GeoDataFrame(
    ortho_comb,
    geometry=gpd.points_from_xy(ortho_comb['easting'], ortho_comb['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)


# Pontos ORTHO combinados (preto)
if not gdf_ortho_comb.empty:
    gdf_ortho_comb.plot(ax=ax_map, color='black', markersize=25, alpha=0.8, label='Pontos ORTHO')

# IDs das células
for _, row in grid_sel.iterrows():
    x_min, y_min, x_max, y_max = row['geometry'].bounds
    ax_map.text(
        x_min, y_max, row['cell_id'],
        fontsize=9, ha='left', va='top', color='black',
        bbox=dict(facecolor='white', alpha=0.6, pad=1)
    )

# Adicionar basemap e formato
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title("Mapa das Células Selecionadas com Pontos ASC/DESC e ORTHO", fontsize=14)
ax_map.set_axis_off()

# Legenda organizada (inclui o tamanho da grelha)
ax_map.legend(loc='lower left', fontsize=9, frameon=True)

plt.show()


# ==============================
# 13. Grid de gráficos - células selecionadas
# ==============================

# Definir células que queres plotar (exemplo)
selected_ids_plot = ["3_5","4_5","5_5","6_5","7_5",
                "3_4","4_4","5_4","6_4",
                "5_6","6_6","7_6"]  # coloca as que quiseres

# Calcular limites globais apenas dessas células
v_min = min(
    agg_sel[agg_sel['cell_id'].isin(selected_ids_plot)]['dV'].min(),
    agg_sel[agg_sel['cell_id'].isin(selected_ids_plot)]['dH'].min(),
    ortho_v_sel[ortho_v_sel['cell_id'].isin(selected_ids_plot)]['disp'].min(),
    ortho_h_sel[ortho_h_sel['cell_id'].isin(selected_ids_plot)]['disp'].min()
)
v_max = max(
    agg_sel[agg_sel['cell_id'].isin(selected_ids_plot)]['dV'].max(),
    agg_sel[agg_sel['cell_id'].isin(selected_ids_plot)]['dH'].max(),
    ortho_v_sel[ortho_v_sel['cell_id'].isin(selected_ids_plot)]['disp'].max(),
    ortho_h_sel[ortho_h_sel['cell_id'].isin(selected_ids_plot)]['disp'].max()
)

# Número de células
n_cells = len(selected_ids_plot)
n_cols = 4  # podes ajustar
n_rows = int(np.ceil(n_cells / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows*5), sharex=True, sharey=True)
if n_rows == 1:
    axes = axes.reshape(1, -1)

for i, cid in enumerate(selected_ids_plot):
    r = i // n_cols
    c = i % n_cols
    ax = axes[r, c]

    agg_cell = agg_sel[agg_sel['cell_id'] == cid]
    ortho_v_cell = ortho_v_sel[ortho_v_sel['cell_id'] == cid]
    ortho_h_cell = ortho_h_sel[ortho_h_sel['cell_id'] == cid]

    # ASC/DESC
    ax.plot(agg_cell['date'], agg_cell['dV'], color='blue', label='ASC/DESC dV')
    ax.plot(agg_cell['date'], agg_cell['dH'], color='green', label='ASC/DESC dH')

    # ORTHO (tracejado)
    if not ortho_v_cell.empty:
        ax.plot(ortho_v_cell['date'], ortho_v_cell['disp'], color='blue', linestyle='--', label='ORTHO dV')
    if not ortho_h_cell.empty:
        ax.plot(ortho_h_cell['date'], ortho_h_cell['disp'], color='green', linestyle='--', label='ORTHO dH')

    ax.set_title(f"Célula {cid}", fontsize=10)
    ax.set_ylabel("Deslocamento (mm)")
    ax.set_xlabel("Data")
    ax.set_ylim(v_min, v_max)
    ax.legend(fontsize=8, loc='best')

# Remover eixos vazios
for j in range(i+1, n_rows*n_cols):
    r = j // n_cols
    c = j % n_cols
    fig.delaxes(axes[r, c])

plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
from scipy.stats import pearsonr

# ==============================
# 14. Correlação InSAR vs ORTHO por célula (selecionadas)
# ==============================

# Células que queres plotar (mesmas do passo 13)
selected_ids_plot = ["3_5","4_5","5_5","6_5","7_5",
                "3_4","4_4","5_4","6_4",
                "5_6","6_6","7_6"]  # substitui pelas que quiseres

n_cells = len(selected_ids_plot)
n_cols = 3  # ajustar conforme layout
n_rows = int(np.ceil(n_cells / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows*5), sharex=True, sharey=True)
if n_rows == 1:
    axes = axes.reshape(1, -1)

for i, cid in enumerate(selected_ids_plot):
    r = i // n_cols
    c = i % n_cols
    ax = axes[r, c]

    # Extrair dados da célula
    agg_cell = agg_sel[agg_sel['cell_id'] == cid]
    ortho_v_cell = ortho_v_sel[ortho_v_sel['cell_id'] == cid]
    ortho_h_cell = ortho_h_sel[ortho_h_sel['cell_id'] == cid]

    # Combinar séries (InSAR vs ORTHO)
    merged_v = pd.merge(
        agg_cell[['date', 'dV']],
        ortho_v_cell[['date', 'disp']],
        on='date',
        how='inner'
    ).rename(columns={'dV': 'disp_insar', 'disp': 'disp_ortho'})

    merged_h = pd.merge(
        agg_cell[['date', 'dH']],
        ortho_h_cell[['date', 'disp']],
        on='date',
        how='inner'
    ).rename(columns={'dH': 'disp_insar', 'disp': 'disp_ortho'})

    # Calcular correlação
    r_v = merged_v['disp_insar'].corr(merged_v['disp_ortho']) if len(merged_v) > 1 else np.nan
    r_h = merged_h['disp_insar'].corr(merged_h['disp_ortho']) if len(merged_h) > 1 else np.nan

    # Scatter plot
    if not merged_v.empty:
        ax.scatter(merged_v['disp_ortho'], merged_v['disp_insar'], color='blue', alpha=0.6, label=f'dV (r={r_v:.2f})')
    if not merged_h.empty:
        ax.scatter(merged_h['disp_ortho'], merged_h['disp_insar'], color='green', alpha=0.6, label=f'dH (r={r_h:.2f})')

    # Reta identidade
    if not merged_v.empty or not merged_h.empty:
        vmin = np.nanmin([
            merged_v['disp_ortho'].min() if not merged_v.empty else np.nan,
            merged_h['disp_ortho'].min() if not merged_h.empty else np.nan
        ])
        vmax = np.nanmax([
            merged_v['disp_ortho'].max() if not merged_v.empty else np.nan,
            merged_h['disp_ortho'].max() if not merged_h.empty else np.nan
        ])
        ax.plot([vmin, vmax], [vmin, vmax], 'k--', lw=1)

    ax.set_title(f"Célula {cid}", fontsize=10)
    ax.set_xlabel("ORTHO (mm)")
    ax.set_ylabel("InSAR (mm)")
    ax.legend(fontsize=8, loc='best')
    ax.grid(False)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)

# Apagar eixos vazios
for j in range(i+1, n_rows*n_cols):
    r = j // n_cols
    c = j % n_cols
    fig.delaxes(axes[r, c])

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# ==============================
# Preparar dados globais apenas para células selecionadas
# ==============================
merged_v_list = []
merged_h_list = []

for cid in selected_ids:
    # Filtrar apenas a célula selecionada
    agg_cell = agg_all[agg_all['cell_id'] == cid]
    ortho_v_cell = ortho_v_long[ortho_v_long['cell_id'] == cid]
    ortho_h_cell = ortho_h_long[ortho_h_long['cell_id'] == cid]

    # Merge vertical usando cell_id e date
    merged_v = pd.merge(
        agg_cell[['cell_id','date','dV']],
        ortho_v_cell[['cell_id','date','disp']],
        on=['cell_id','date'],
        how='inner'
    ).rename(columns={'dV':'disp_insar', 'disp':'disp_ortho'})

    # Merge horizontal usando cell_id e date
    merged_h = pd.merge(
        agg_cell[['cell_id','date','dH']],
        ortho_h_cell[['cell_id','date','disp']],
        on=['cell_id','date'],
        how='inner'
    ).rename(columns={'dH':'disp_insar', 'disp':'disp_ortho'})

    merged_v_list.append(merged_v)
    merged_h_list.append(merged_h)

# Concatenar apenas células selecionadas
merged_v_global = pd.concat(merged_v_list, ignore_index=True)
merged_h_global = pd.concat(merged_h_list, ignore_index=True)

# ==============================
# Calcular métricas
# ==============================
def compute_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    r_corr = np.corrcoef(y_true, y_pred)[0,1]
    return rmse, mae, r2, r_corr

rmse_v, mae_v, r2_v, r_v = compute_metrics(merged_v_global['disp_ortho'], merged_v_global['disp_insar'])
rmse_h, mae_h, r2_h, r_h = compute_metrics(merged_h_global['disp_ortho'], merged_h_global['disp_insar'])

# ==============================
# Gráfico lado a lado
# ==============================
fig, axes = plt.subplots(1, 2, figsize=(14,6))

# --- dV ---
ax = axes[0]
ax.scatter(merged_v_global['disp_ortho'], merged_v_global['disp_insar'], color='blue', alpha=0.3)
vmin_v = min(merged_v_global['disp_ortho'].min(), merged_v_global['disp_insar'].min())
vmax_v = max(merged_v_global['disp_ortho'].max(), merged_v_global['disp_insar'].max())
ax.plot([vmin_v, vmax_v], [vmin_v, vmax_v], 'k--', lw=1)
ax.set_xlabel("ORTHO dV (mm)")
ax.set_ylabel("InSAR dV (mm)")
ax.set_title("Comparação dV InSAR vs ORTHO")
textstr_v = f"""r = {r_v:.2f}
RMSE = {rmse_v:.2f} mm
MAE = {mae_v:.2f} mm
R² = {r2_v:.2f}"""
ax.text(0.05, 0.95, textstr_v, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# --- dH ---
ax = axes[1]
ax.scatter(merged_h_global['disp_ortho'], merged_h_global['disp_insar'], color='green', alpha=0.3)
vmin_h = min(merged_h_global['disp_ortho'].min(), merged_h_global['disp_insar'].min())
vmax_h = max(merged_h_global['disp_ortho'].max(), merged_h_global['disp_insar'].max())
ax.plot([vmin_h, vmax_h], [vmin_h, vmax_h], 'k--', lw=1)
ax.set_xlabel("ORTHO dH (mm)")
ax.set_ylabel("InSAR dH (mm)")
ax.set_title("Comparação dH InSAR vs ORTHO")
textstr_h = f"""r = {r_h:.2f}
RMSE = {rmse_h:.2f} mm
MAE = {mae_h:.2f} mm
R² = {r2_h:.2f}"""
ax.text(0.05, 0.95, textstr_h, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()



In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import seaborn as sns

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

agg = asc_interp.dropna(subset=['cell_x','cell_y']).groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Selecionar células
# ==============================
selected_ids_plot = ["3_5","4_5","5_5","6_5","7_5",
                     "3_4","4_4","5_4","6_4",
                     "5_6","6_6","7_6"]  # células de interesse

agg_sel = agg[agg['cell_id'].isin(selected_ids_plot)]

# Atribuir células ORTHO
def melt_ortho(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing'], value_vars=disp_cols,
                      var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

ortho_v_long = melt_ortho(ortho_v)
ortho_h_long = melt_ortho(ortho_h)

for ortho_df in [ortho_v_long, ortho_h_long]:
    ortho_df['cell_x'] = pd.cut(ortho_df['easting'], bins=x_edges_shifted, labels=False)
    ortho_df['cell_y'] = pd.cut(ortho_df['northing'], bins=y_edges_shifted, labels=False)
    ortho_df.dropna(subset=['cell_x','cell_y'], inplace=True)
    ortho_df['cell_id'] = ortho_df['cell_x'].astype(int).astype(str) + "_" + ortho_df['cell_y'].astype(int).astype(str)

ortho_v_sel = ortho_v_long[ortho_v_long['cell_id'].isin(selected_ids_plot)]
ortho_h_sel = ortho_h_long[ortho_h_long['cell_id'].isin(selected_ids_plot)]

# ==============================
# 10. Scatter e métricas globais para células selecionadas
# ==============================
merged_v_list = []
merged_h_list = []

for cid in selected_ids_plot:
    agg_cell = agg_sel[agg_sel['cell_id'] == cid]
    ortho_v_cell = ortho_v_sel[ortho_v_sel['cell_id'] == cid]
    ortho_h_cell = ortho_h_sel[ortho_h_sel['cell_id'] == cid]

    merged_v = pd.merge(agg_cell[['cell_id','date','dV']],
                        ortho_v_cell[['cell_id','date','disp']],
                        on=['cell_id','date'], how='inner').rename(columns={'dV':'disp_insar','disp':'disp_ortho'})
    merged_h = pd.merge(agg_cell[['cell_id','date','dH']],
                        ortho_h_cell[['cell_id','date','disp']],
                        on=['cell_id','date'], how='inner').rename(columns={'dH':'disp_insar','disp':'disp_ortho'})
    merged_v_list.append(merged_v)
    merged_h_list.append(merged_h)

merged_v_global = pd.concat(merged_v_list, ignore_index=True)
merged_h_global = pd.concat(merged_h_list, ignore_index=True)

def compute_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    r_corr = np.corrcoef(y_true, y_pred)[0,1]
    return rmse, mae, r2, r_corr

rmse_v, mae_v, r2_v, r_v = compute_metrics(merged_v_global['disp_ortho'], merged_v_global['disp_insar'])
rmse_h, mae_h, r2_h, r_h = compute_metrics(merged_h_global['disp_ortho'], merged_h_global['disp_insar'])

fig, axes = plt.subplots(1, 2, figsize=(14,6))

ax = axes[0]
ax.scatter(merged_v_global['disp_ortho'], merged_v_global['disp_insar'], color='blue', alpha=0.3)
vmin_v, vmax_v = merged_v_global['disp_ortho'].min(), merged_v_global['disp_ortho'].max()
ax.plot([vmin_v, vmax_v],[vmin_v, vmax_v],'k--')
ax.set_xlabel("ORTHO dV (mm)")
ax.set_ylabel("InSAR dV (mm)")
ax.set_title("Comparação dV InSAR vs ORTHO")
ax.text(0.05,0.95,f"r={r_v:.2f}\nRMSE={rmse_v:.2f}\nMAE={mae_v:.2f}\nR²={r2_v:.2f}", transform=ax.transAxes,
        verticalalignment='top', bbox=dict(facecolor='white', alpha=0.8))

ax = axes[1]
ax.scatter(merged_h_global['disp_ortho'], merged_h_global['disp_insar'], color='green', alpha=0.3)
vmin_h, vmax_h = merged_h_global['disp_ortho'].min(), merged_h_global['disp_ortho'].max()
ax.plot([vmin_h, vmax_h],[vmin_h, vmax_h],'k--')
ax.set_xlabel("ORTHO dH (mm)")
ax.set_ylabel("InSAR dH (mm)")
ax.set_title("Comparação dH InSAR vs ORTHO")
ax.text(0.05,0.95,f"r={r_h:.2f}\nRMSE={rmse_h:.2f}\nMAE={mae_h:.2f}\nR²={r2_h:.2f}", transform=ax.transAxes,
        verticalalignment='top', bbox=dict(facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()


In [ ]:
# =======================================
# Script completo: Processamento e Visualizações
# =======================================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import seaborn as sns

# =======================================
# 1. Ler CSVs ASC, DESC e ORTHO
# =======================================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# =======================================
# 2. Filtrar área de interesse
# =======================================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# =======================================
# 3. Melt para ASC/DESC
# =======================================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
                      value_vars=disp_cols, var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(start=max(asc_long['date'].min(), desc_long['date'].min()),
                             end=min(asc_long['date'].max(), desc_long['date'].max()),
                             freq='MS')

# =======================================
# 4. Interpolação temporal linear
# =======================================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64),
                           group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates, 'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# =======================================
# 5. Interpolação espacial IDW
# =======================================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# =======================================
# 6. Calcular β e γ
# =======================================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# =======================================
# 7. Calcular dV e dH
# =======================================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) + np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) + dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# =======================================
# 8. Grelha centrada ORTHO
# =======================================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

agg = asc_interp.dropna(subset=['cell_x','cell_y']).groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# =======================================
# 9. Selecionar células
# =======================================
selected_ids_plot = ["3_5","4_5","5_5","6_5","7_5","3_4","4_4","5_4","6_4","5_6","6_6","7_6"]
agg_sel = agg[agg['cell_id'].isin(selected_ids_plot)]

def melt_ortho(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing'], value_vars=disp_cols,
                      var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

ortho_v_long = melt_ortho(ortho_v)
ortho_h_long = melt_ortho(ortho_h)

for ortho_df in [ortho_v_long, ortho_h_long]:
    ortho_df['cell_x'] = pd.cut(ortho_df['easting'], bins=x_edges_shifted, labels=False)
    ortho_df['cell_y'] = pd.cut(ortho_df['northing'], bins=y_edges_shifted, labels=False)
    ortho_df.dropna(subset=['cell_x','cell_y'], inplace=True)
    ortho_df['cell_id'] = ortho_df['cell_x'].astype(int).astype(str) + "_" + ortho_df['cell_y'].astype(int).astype(str)

ortho_v_sel = ortho_v_long[ortho_v_long['cell_id'].isin(selected_ids_plot)]
ortho_h_sel = ortho_h_long[ortho_h_long['cell_id'].isin(selected_ids_plot)]

# =======================================
# 10. Figura 1: Mapa das células e pontos
# =======================================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({"cell_x": ix, "cell_y": iy, "cell_id": f"{ix}_{iy}",
                          "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                                          x_edges_shifted[ix+1], y_edges_shifted[iy+1])})
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)
points = agg.groupby('cell_id').first().reset_index()
gdf_points = gpd.GeoDataFrame(points, geometry=gpd.points_from_xy(points['x_center'], points['y_center']),
                              crs="EPSG:3035").to_crs(epsg=3857)
grid_sel = grid[grid['cell_id'].isin(selected_ids_plot)]
points_sel = gdf_points[gdf_points['cell_id'].isin(selected_ids_plot)]

fig_map, ax_map = plt.subplots(figsize=(10,10))
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5, label=f'Grelha Base ({grid_size} m)')
grid_sel.boundary.plot(ax=ax_map, color='red', linewidth=1.5, label=f'Células Selecionadas ({grid_size} m)')
points_sel.plot(ax=ax_map, color='white', edgecolor='black', markersize=60, label='Pontos ASC/DESC')

ortho_comb = pd.concat([ortho_v_sel, ortho_h_sel], ignore_index=True).drop_duplicates(subset=['easting','northing','date'])
gdf_ortho_comb = gpd.GeoDataFrame(ortho_comb, geometry=gpd.points_from_xy(ortho_comb['easting'], ortho_comb['northing']),
                                  crs="EPSG:3035").to_crs(epsg=3857)
if not gdf_ortho_comb.empty:
    gdf_ortho_comb.plot(ax=ax_map, color='black', markersize=25, alpha=0.8, label='Pontos ORTHO')

for _, row in grid_sel.iterrows():
    x_min, y_min, x_max, y_max = row['geometry'].bounds
    ax_map.text(x_min, y_max, row['cell_id'], fontsize=9, ha='left', va='top',
                bbox=dict(facecolor='white', alpha=0.6, pad=1))
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title("Mapa das Células Selecionadas", fontsize=14)
ax_map.set_axis_off()
ax_map.legend(loc='lower left', fontsize=9)
plt.show()

# =======================================
# 11. Figura 2: Séries temporais InSAR vs ORTHO
# =======================================
n_cells = len(selected_ids_plot)
n_cols = 4
n_rows = int(np.ceil(n_cells / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows*4), sharex=True, sharey=True)
if n_rows==1: axes=axes.reshape(1,-1)

v_min = min(agg_sel['dV'].min(), agg_sel['dH'].min(),
            ortho_v_sel['disp'].min(), ortho_h_sel['disp'].min())
v_max = max(agg_sel['dV'].max(), agg_sel['dH'].max(),
            ortho_v_sel['disp'].max(), ortho_h_sel['disp'].max())

for i, cid in enumerate(selected_ids_plot):
    r, c = i // n_cols, i % n_cols
    ax = axes[r, c]
    agg_cell = agg_sel[agg_sel['cell_id']==cid]
    ortho_v_cell = ortho_v_sel[ortho_v_sel['cell_id']==cid]
    ortho_h_cell = ortho_h_sel[ortho_h_sel['cell_id']==cid]

    ax.plot(agg_cell['date'], agg_cell['dV'], color='blue', label='ASC/DESC dV')
    ax.plot(agg_cell['date'], agg_cell['dH'], color='green', label='ASC/DESC dH')
    if not ortho_v_cell.empty: ax.plot(ortho_v_cell['date'], ortho_v_cell['disp'], color='blue', linestyle='--', label='ORTHO dV')
    if not ortho_h_cell.empty: ax.plot(ortho_h_cell['date'], ortho_h_cell['disp'], color='green', linestyle='--', label='ORTHO dH')
    ax.set_title(f"Célula {cid}", fontsize=10)
    ax.set_xlabel("Data")
    ax.set_ylabel("Deslocamento (mm)")
    ax.set_ylim(v_min,v_max)
    ax.legend(fontsize=8, loc='best')

for j in range(i+1, n_rows*n_cols):
    r, c = j // n_cols, j % n_cols
    fig.delaxes(axes[r,c])
plt.tight_layout()
plt.show()

# =======================================
# 12. Figura 3: Scatter InSAR vs ORTHO por célula
# =======================================
n_cols = 3
n_rows = int(np.ceil(n_cells/n_cols))
fig, axes = plt.subplots(n_rows,n_cols,figsize=(15,n_rows*5), sharex=True, sharey=True)
if n_rows==1: axes=axes.reshape(1,-1)

for i, cid in enumerate(selected_ids_plot):
    r,c = i//n_cols,i%n_cols
    ax = axes[r,c]
    agg_cell = agg_sel[agg_sel['cell_id']==cid]
    ortho_v_cell = ortho_v_sel[ortho_v_sel['cell_id']==cid]
    ortho_h_cell = ortho_h_sel[ortho_h_sel['cell_id']==cid]

    merged_v = pd.merge(agg_cell[['date','dV']], ortho_v_cell[['date','disp']], on='date', how='inner').rename(columns={'dV':'disp_insar','disp':'disp_ortho'})
    merged_h = pd.merge(agg_cell[['date','dH']], ortho_h_cell[['date','disp']], on='date', how='inner').rename(columns={'dH':'disp_insar','disp':'disp_ortho'})

    r_v = merged_v['disp_insar'].corr(merged_v['disp_ortho']) if len(merged_v)>1 else np.nan
    r_h = merged_h['disp_insar'].corr(merged_h['disp_ortho']) if len(merged_h)>1 else np.nan

    if not merged_v.empty: ax.scatter(merged_v['disp_ortho'], merged_v['disp_insar'], color='blue', alpha=0.6, label=f'dV (r={r_v:.2f})')
    if not merged_h.empty: ax.scatter(merged_h['disp_ortho'], merged_h['disp_insar'], color='green', alpha=0.6, label=f'dH (r={r_h:.2f})')

    vmin = np.nanmin([merged_v['disp_ortho'].min() if not merged_v.empty else np.nan,
                      merged_h['disp_ortho'].min() if not merged_h.empty else np.nan])
    vmax = np.nanmax([merged_v['disp_ortho'].max() if not merged_v.empty else np.nan,
                      merged_h['disp_ortho'].max() if not merged_h.empty else np.nan])
    if not np.isnan(vmin) and not np.isnan(vmax): ax.plot([vmin,vmax],[vmin,vmax],'k--', lw=1)

    ax.set_title(f"Célula {cid}", fontsize=10)
    ax.set_xlabel("ORTHO (mm)")
    ax.set_ylabel("InSAR (mm)")
    ax.legend(fontsize=8, loc='best')
    ax.grid(False)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)

for j in range(i+1, n_rows*n_cols):
    r,c = j//n_cols,j%n_cols
    fig.delaxes(axes[r,c])
plt.tight_layout()
plt.show()

# =======================================
# 13. Figura 4: Scatter global com métricas
# =======================================
merged_v_list = []
merged_h_list = []
for cid in selected_ids_plot:
    agg_cell = agg_sel[agg_sel['cell_id']==cid]
    ortho_v_cell = ortho_v_sel[ortho_v_sel['cell_id']==cid]
    ortho_h_cell = ortho_h_sel[ortho_h_sel['cell_id']==cid]
    merged_v_list.append(pd.merge(agg_cell[['cell_id','date','dV']], ortho_v_cell[['cell_id','date','disp']], on=['cell_id','date'], how='inner').rename(columns={'dV':'disp_insar','disp':'disp_ortho'}))
    merged_h_list.append(pd.merge(agg_cell[['cell_id','date','dH']], ortho_h_cell[['cell_id','date','disp']], on=['cell_id','date'], how='inner').rename(columns={'dH':'disp_insar','disp':'disp_ortho'}))

merged_v_global = pd.concat(merged_v_list, ignore_index=True)
merged_h_global = pd.concat(merged_h_list, ignore_index=True)

def compute_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    r_corr = np.corrcoef(y_true, y_pred)[0,1]
    return rmse, mae, r2, r_corr

rmse_v, mae_v, r2_v, r_v = compute_metrics(merged_v_global['disp_ortho'], merged_v_global['disp_insar'])
rmse_h, mae_h, r2_h, r_h = compute_metrics(merged_h_global['disp_ortho'], merged_h_global['disp_insar'])

fig, axes = plt.subplots(1,2,figsize=(14,6))
axes[0].scatter(merged_v_global['disp_ortho'], merged_v_global['disp_insar'], color='blue', alpha=0.3)
axes[0].plot([merged_v_global['disp_ortho'].min(), merged_v_global['disp_ortho'].max()],
             [merged_v_global['disp_ortho'].min(), merged_v_global['disp_ortho'].max()],'k--')
axes[0].set_xlabel("ORTHO dV (mm)")
axes[0].set_ylabel("InSAR dV (mm)")
axes[0].set_title("Comparação dV InSAR vs ORTHO")
axes[0].text(0.05,0.95,f"r={r_v:.2f}\nRMSE={rmse_v:.2f}\nMAE={mae_v:.2f}\nR²={r2_v:.2f}", transform=axes[0].transAxes,
             verticalalignment='top', bbox=dict(facecolor='white', alpha=0.8))

axes[1].scatter(merged_h_global['disp_ortho'], merged_h_global['disp_insar'], color='green', alpha=0.3)
axes[1].plot([merged_h_global['disp_ortho'].min(), merged_h_global['disp_ortho'].max()],
             [merged_h_global['disp_ortho'].min(), merged_h_global['disp_ortho'].max()],'k--')
axes[1].set_xlabel("ORTHO dH (mm)")
axes[1].set_ylabel("InSAR dH (mm)")
axes[1].set_title("Comparação dH InSAR vs ORTHO")
axes[1].text(0.05,0.95,f"r={r_h:.2f}\nRMSE={rmse_h:.2f}\nMAE={mae_h:.2f}\nR²={r2_h:.2f}", transform=axes[1].transAxes,
             verticalalignment='top', bbox=dict(facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()


In [ ]:
# =======================================
# Script completo: Processamento e Visualizações
# =======================================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import seaborn as sns

# =======================================
# 1. Ler CSVs ASC, DESC e ORTHO
# =======================================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# =======================================
# 2. Filtrar área de interesse
# =======================================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# =======================================
# 3. Melt para ASC/DESC
# =======================================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
                      value_vars=disp_cols, var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(start=max(asc_long['date'].min(), desc_long['date'].min()),
                             end=min(asc_long['date'].max(), desc_long['date'].max()),
                             freq='MS')

# =======================================
# 4. Interpolação temporal linear
# =======================================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64),
                           group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates, 'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# =======================================
# 5. Interpolação espacial IDW
# =======================================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# =======================================
# 6. Calcular β e γ
# =======================================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# =======================================
# 7. Calcular dV e dH
# =======================================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) + np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) + dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# =======================================
# 8. Grelha centrada ORTHO
# =======================================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

agg = asc_interp.dropna(subset=['cell_x','cell_y']).groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# =======================================
# 9. Selecionar células
# =======================================
selected_ids_plot = ["3_5","4_5","5_5","6_5","7_5","3_4","4_4","5_4","6_4","5_6","6_6","7_6"]
agg_sel = agg[agg['cell_id'].isin(selected_ids_plot)]

def melt_ortho(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing'], value_vars=disp_cols,
                      var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

ortho_v_long = melt_ortho(ortho_v)
ortho_h_long = melt_ortho(ortho_h)

for ortho_df in [ortho_v_long, ortho_h_long]:
    ortho_df['cell_x'] = pd.cut(ortho_df['easting'], bins=x_edges_shifted, labels=False)
    ortho_df['cell_y'] = pd.cut(ortho_df['northing'], bins=y_edges_shifted, labels=False)
    ortho_df.dropna(subset=['cell_x','cell_y'], inplace=True)
    ortho_df['cell_id'] = ortho_df['cell_x'].astype(int).astype(str) + "_" + ortho_df['cell_y'].astype(int).astype(str)

ortho_v_sel = ortho_v_long[ortho_v_long['cell_id'].isin(selected_ids_plot)]
ortho_h_sel = ortho_h_long[ortho_h_long['cell_id'].isin(selected_ids_plot)]

# =======================================
# 10. Figura 1: Mapa das células e pontos
# =======================================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({"cell_x": ix, "cell_y": iy, "cell_id": f"{ix}_{iy}",
                          "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                                          x_edges_shifted[ix+1], y_edges_shifted[iy+1])})
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)
points = agg.groupby('cell_id').first().reset_index()
gdf_points = gpd.GeoDataFrame(points, geometry=gpd.points_from_xy(points['x_center'], points['y_center']),
                              crs="EPSG:3035").to_crs(epsg=3857)
grid_sel = grid[grid['cell_id'].isin(selected_ids_plot)]
points_sel = gdf_points[gdf_points['cell_id'].isin(selected_ids_plot)]

fig_map, ax_map = plt.subplots(figsize=(10,10))
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5, label=f'Grelha Base ({grid_size} m)')
grid_sel.boundary.plot(ax=ax_map, color='red', linewidth=1.5, label=f'Células Selecionadas ({grid_size} m)')
points_sel.plot(ax=ax_map, color='white', edgecolor='black', markersize=60, label='Pontos ASC/DESC')

ortho_comb = pd.concat([ortho_v_sel, ortho_h_sel], ignore_index=True).drop_duplicates(subset=['easting','northing','date'])
gdf_ortho_comb = gpd.GeoDataFrame(ortho_comb, geometry=gpd.points_from_xy(ortho_comb['easting'], ortho_comb['northing']),
                                  crs="EPSG:3035").to_crs(epsg=3857)
if not gdf_ortho_comb.empty:
    gdf_ortho_comb.plot(ax=ax_map, color='black', markersize=25, alpha=0.8, label='Pontos ORTHO')

for _, row in grid_sel.iterrows():
    x_min, y_min, x_max, y_max = row['geometry'].bounds
    ax_map.text(x_min, y_max, row['cell_id'], fontsize=9, ha='left', va='top',
                bbox=dict(facecolor='white', alpha=0.6, pad=1))
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title("Mapa das Células Selecionadas", fontsize=14)
ax_map.set_axis_off()
ax_map.legend(loc='lower left', fontsize=9)
plt.show()

# =======================================
# 11. Figura 2: Séries temporais InSAR vs ORTHO
# =======================================
n_cells = len(selected_ids_plot)
n_cols = 3
n_rows = int(np.ceil(n_cells / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows*3), sharex=True, sharey=True)
if n_rows==1: axes=axes.reshape(1,-1)

v_min = min(agg_sel['dV'].min(), agg_sel['dH'].min(),
            ortho_v_sel['disp'].min(), ortho_h_sel['disp'].min())
v_max = max(agg_sel['dV'].max(), agg_sel['dH'].max(),
            ortho_v_sel['disp'].max(), ortho_h_sel['disp'].max())

for i, cid in enumerate(selected_ids_plot):
    r, c = i // n_cols, i % n_cols
    ax = axes[r, c]
    agg_cell = agg_sel[agg_sel['cell_id']==cid]
    ortho_v_cell = ortho_v_sel[ortho_v_sel['cell_id']==cid]
    ortho_h_cell = ortho_h_sel[ortho_h_sel['cell_id']==cid]

    ax.plot(agg_cell['date'], agg_cell['dV'], color='blue', label='ASC/DESC dV')
    ax.plot(agg_cell['date'], agg_cell['dH'], color='green', label='ASC/DESC dH')
    if not ortho_v_cell.empty: ax.plot(ortho_v_cell['date'], ortho_v_cell['disp'], color='blue', linestyle='--', label='ORTHO dV')
    if not ortho_h_cell.empty: ax.plot(ortho_h_cell['date'], ortho_h_cell['disp'], color='green', linestyle='--', label='ORTHO dH')
    ax.set_title(f"Célula {cid}", fontsize=10)
    ax.set_xlabel("Data")
    ax.set_ylabel("Deslocamento (mm)")
    ax.set_ylim(v_min,v_max)
    ax.legend(fontsize=8, loc='best')

for j in range(i+1, n_rows*n_cols):
    r, c = j // n_cols, j % n_cols
    fig.delaxes(axes[r,c])
plt.tight_layout()
plt.show()

# =======================================
# 12. Figura 3: Scatter InSAR vs ORTHO por célula
# =======================================
n_cols = 3
n_rows = int(np.ceil(n_cells/n_cols))
fig, axes = plt.subplots(n_rows,n_cols,figsize=(15,n_rows*3), sharex=True, sharey=True)
if n_rows==1: axes=axes.reshape(1,-1)

for i, cid in enumerate(selected_ids_plot):
    r,c = i//n_cols,i%n_cols
    ax = axes[r,c]
    agg_cell = agg_sel[agg_sel['cell_id']==cid]
    ortho_v_cell = ortho_v_sel[ortho_v_sel['cell_id']==cid]
    ortho_h_cell = ortho_h_sel[ortho_h_sel['cell_id']==cid]

    merged_v = pd.merge(agg_cell[['date','dV']], ortho_v_cell[['date','disp']], on='date', how='inner').rename(columns={'dV':'disp_insar','disp':'disp_ortho'})
    merged_h = pd.merge(agg_cell[['date','dH']], ortho_h_cell[['date','disp']], on='date', how='inner').rename(columns={'dH':'disp_insar','disp':'disp_ortho'})

    r_v = merged_v['disp_insar'].corr(merged_v['disp_ortho']) if len(merged_v)>1 else np.nan
    r_h = merged_h['disp_insar'].corr(merged_h['disp_ortho']) if len(merged_h)>1 else np.nan

    if not merged_v.empty: ax.scatter(merged_v['disp_ortho'], merged_v['disp_insar'], color='blue', alpha=0.6, label=f'dV (r={r_v:.2f})')
    if not merged_h.empty: ax.scatter(merged_h['disp_ortho'], merged_h['disp_insar'], color='green', alpha=0.6, label=f'dH (r={r_h:.2f})')

    vmin = np.nanmin([merged_v['disp_ortho'].min() if not merged_v.empty else np.nan,
                      merged_h['disp_ortho'].min() if not merged_h.empty else np.nan])
    vmax = np.nanmax([merged_v['disp_ortho'].max() if not merged_v.empty else np.nan,
                      merged_h['disp_ortho'].max() if not merged_h.empty else np.nan])
    if not np.isnan(vmin) and not np.isnan(vmax): ax.plot([vmin,vmax],[vmin,vmax],'k--', lw=1)

    ax.set_title(f"Célula {cid}", fontsize=10)
    ax.set_xlabel("ORTHO (mm)")
    ax.set_ylabel("InSAR (mm)")
    ax.legend(fontsize=8, loc='best')
    ax.grid(False)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)

for j in range(i+1, n_rows*n_cols):
    r,c = j//n_cols,j%n_cols
    fig.delaxes(axes[r,c])
plt.tight_layout()
plt.show()

# =======================================
# 13. Figura 4: Scatter global com métricas
# =======================================
merged_v_list = []
merged_h_list = []
for cid in selected_ids_plot:
    agg_cell = agg_sel[agg_sel['cell_id']==cid]
    ortho_v_cell = ortho_v_sel[ortho_v_sel['cell_id']==cid]
    ortho_h_cell = ortho_h_sel[ortho_h_sel['cell_id']==cid]
    merged_v_list.append(pd.merge(agg_cell[['cell_id','date','dV']], ortho_v_cell[['cell_id','date','disp']], on=['cell_id','date'], how='inner').rename(columns={'dV':'disp_insar','disp':'disp_ortho'}))
    merged_h_list.append(pd.merge(agg_cell[['cell_id','date','dH']], ortho_h_cell[['cell_id','date','disp']], on=['cell_id','date'], how='inner').rename(columns={'dH':'disp_insar','disp':'disp_ortho'}))

merged_v_global = pd.concat(merged_v_list, ignore_index=True)
merged_h_global = pd.concat(merged_h_list, ignore_index=True)

def compute_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    r_corr = np.corrcoef(y_true, y_pred)[0,1]
    return rmse, mae, r2, r_corr



rmse_v, mae_v, r2_v, r_v = compute_metrics(merged_v_global['disp_ortho'], merged_v_global['disp_insar'])
rmse_h, mae_h, r2_h, r_h = compute_metrics(merged_h_global['disp_ortho'], merged_h_global['disp_insar'])

fig, axes = plt.subplots(1,2,figsize=(14,6))
axes[0].scatter(merged_v_global['disp_ortho'], merged_v_global['disp_insar'], color='blue', alpha=0.3)
axes[0].plot([merged_v_global['disp_ortho'].min(), merged_v_global['disp_ortho'].max()],
             [merged_v_global['disp_ortho'].min(), merged_v_global['disp_ortho'].max()],'k--')
axes[0].set_xlabel("ORTHO dV (mm)")
axes[0].set_ylabel("InSAR dV (mm)")
axes[0].set_title("Comparação dV InSAR vs ORTHO")
axes[0].text(0.05,0.95,f"r={r_v:.2f}\nRMSE={rmse_v:.2f}\nMAE={mae_v:.2f}\nR²={r2_v:.2f}", transform=axes[0].transAxes,
             verticalalignment='top', bbox=dict(facecolor='white', alpha=0.8))

axes[1].scatter(merged_h_global['disp_ortho'], merged_h_global['disp_insar'], color='green', alpha=0.3)
axes[1].plot([merged_h_global['disp_ortho'].min(), merged_h_global['disp_ortho'].max()],
             [merged_h_global['disp_ortho'].min(), merged_h_global['disp_ortho'].max()],'k--')
axes[1].set_xlabel("ORTHO dH (mm)")
axes[1].set_ylabel("InSAR dH (mm)")
axes[1].set_title("Comparação dH InSAR vs ORTHO")
axes[1].text(0.05,0.95,f"r={r_h:.2f}\nRMSE={rmse_h:.2f}\nMAE={mae_h:.2f}\nR²={r2_h:.2f}", transform=axes[1].transAxes,
             verticalalignment='top', bbox=dict(facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

In [ ]:
# =======================================
# Script completo: Processamento e Visualizações
# =======================================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import seaborn as sns

# =======================================
# 1. Ler CSVs ASC, DESC e ORTHO
# =======================================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# =======================================
# 2. Filtrar área de interesse
# =======================================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# =======================================
# 3. Melt para ASC/DESC
# =======================================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
                      value_vars=disp_cols, var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(start=max(asc_long['date'].min(), desc_long['date'].min()),
                             end=min(asc_long['date'].max(), desc_long['date'].max()),
                             freq='MS')

# =======================================
# 4. Interpolação temporal linear
# =======================================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64),
                           group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates, 'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# =======================================
# 5. Interpolação espacial IDW
# =======================================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# =======================================
# 6. Calcular β e γ
# =======================================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# =======================================
# 7. Calcular dV e dH
# =======================================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) + np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) + dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# =======================================
# 8. Grelha centrada ORTHO
# =======================================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

agg = asc_interp.dropna(subset=['cell_x','cell_y']).groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# =======================================
# 9. Selecionar células
# =======================================
selected_ids_plot = ["3_5","4_5","5_5","6_5","7_5","3_4","4_4","5_4","6_4","5_6","6_6","7_6"]
agg_sel = agg[agg['cell_id'].isin(selected_ids_plot)]

def melt_ortho(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing'], value_vars=disp_cols,
                      var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

ortho_v_long = melt_ortho(ortho_v)
ortho_h_long = melt_ortho(ortho_h)

for ortho_df in [ortho_v_long, ortho_h_long]:
    ortho_df['cell_x'] = pd.cut(ortho_df['easting'], bins=x_edges_shifted, labels=False)
    ortho_df['cell_y'] = pd.cut(ortho_df['northing'], bins=y_edges_shifted, labels=False)
    ortho_df.dropna(subset=['cell_x','cell_y'], inplace=True)
    ortho_df['cell_id'] = ortho_df['cell_x'].astype(int).astype(str) + "_" + ortho_df['cell_y'].astype(int).astype(str)

ortho_v_sel = ortho_v_long[ortho_v_long['cell_id'].isin(selected_ids_plot)]
ortho_h_sel = ortho_h_long[ortho_h_long['cell_id'].isin(selected_ids_plot)]

# =======================================
# 10. Figura 1: Mapa das células e pontos
# =======================================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({"cell_x": ix, "cell_y": iy, "cell_id": f"{ix}_{iy}",
                          "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                                          x_edges_shifted[ix+1], y_edges_shifted[iy+1])})
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)
points = agg.groupby('cell_id').first().reset_index()
gdf_points = gpd.GeoDataFrame(points, geometry=gpd.points_from_xy(points['x_center'], points['y_center']),
                              crs="EPSG:3035").to_crs(epsg=3857)
grid_sel = grid[grid['cell_id'].isin(selected_ids_plot)]
points_sel = gdf_points[gdf_points['cell_id'].isin(selected_ids_plot)]

fig_map, ax_map = plt.subplots(figsize=(10,10))
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5, label=f'Grelha Base ({grid_size} m)')
grid_sel.boundary.plot(ax=ax_map, color='red', linewidth=1.5, label=f'Células Selecionadas ({grid_size} m)')
points_sel.plot(ax=ax_map, color='white', edgecolor='black', markersize=60, label='Pontos ASC/DESC')

ortho_comb = pd.concat([ortho_v_sel, ortho_h_sel], ignore_index=True).drop_duplicates(subset=['easting','northing','date'])
gdf_ortho_comb = gpd.GeoDataFrame(ortho_comb, geometry=gpd.points_from_xy(ortho_comb['easting'], ortho_comb['northing']),
                                  crs="EPSG:3035").to_crs(epsg=3857)
if not gdf_ortho_comb.empty:
    gdf_ortho_comb.plot(ax=ax_map, color='black', markersize=25, alpha=0.8, label='Pontos ORTHO')

for _, row in grid_sel.iterrows():
    x_min, y_min, x_max, y_max = row['geometry'].bounds
    ax_map.text(x_min, y_max, row['cell_id'], fontsize=9, ha='left', va='top',
                bbox=dict(facecolor='white', alpha=0.6, pad=1))
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title("Mapa das Células Selecionadas", fontsize=14)
ax_map.set_axis_off()
ax_map.legend(loc='lower left', fontsize=9)
plt.show()

# =======================================
# 11. Figura 2: Séries temporais InSAR vs ORTHO
# =======================================
n_cells = len(selected_ids_plot)
n_cols = 3
n_rows = int(np.ceil(n_cells / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows*3), sharex=True, sharey=True)
if n_rows==1: axes=axes.reshape(1,-1)

v_min = min(agg_sel['dV'].min(), agg_sel['dH'].min(),
            ortho_v_sel['disp'].min(), ortho_h_sel['disp'].min())
v_max = max(agg_sel['dV'].max(), agg_sel['dH'].max(),
            ortho_v_sel['disp'].max(), ortho_h_sel['disp'].max())

for i, cid in enumerate(selected_ids_plot):
    r, c = i // n_cols, i % n_cols
    ax = axes[r, c]
    agg_cell = agg_sel[agg_sel['cell_id']==cid]
    ortho_v_cell = ortho_v_sel[ortho_v_sel['cell_id']==cid]
    ortho_h_cell = ortho_h_sel[ortho_h_sel['cell_id']==cid]

    ax.plot(agg_cell['date'], agg_cell['dV'], color='blue', label='ASC/DESC dV')
    ax.plot(agg_cell['date'], agg_cell['dH'], color='green', label='ASC/DESC dH')
    if not ortho_v_cell.empty: ax.plot(ortho_v_cell['date'], ortho_v_cell['disp'], color='blue', linestyle='--', label='ORTHO dV')
    if not ortho_h_cell.empty: ax.plot(ortho_h_cell['date'], ortho_h_cell['disp'], color='green', linestyle='--', label='ORTHO dH')
    ax.set_title(f"Célula {cid}", fontsize=10)
    ax.set_xlabel("Data")
    ax.set_ylabel("Deslocamento (mm)")
    ax.set_ylim(v_min,v_max)
    ax.legend(fontsize=8, loc='best')

for j in range(i+1, n_rows*n_cols):
    r, c = j // n_cols, j % n_cols
    fig.delaxes(axes[r,c])
plt.tight_layout()
plt.show()

# =======================================
# 12. Figura 3: Scatter InSAR vs ORTHO por célula (com métricas identificadas)
# =======================================
n_cols = 3
n_rows = int(np.ceil(n_cells / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows*3), sharex=True, sharey=True)
if n_rows == 1: 
    axes = axes.reshape(1, -1)

for i, cid in enumerate(selected_ids_plot):
    r, c = i // n_cols, i % n_cols
    ax = axes[r, c]

    # Dados da célula
    agg_cell = agg_sel[agg_sel['cell_id'] == cid]
    ortho_v_cell = ortho_v_sel[ortho_v_sel['cell_id'] == cid]
    ortho_h_cell = ortho_h_sel[ortho_h_sel['cell_id'] == cid]

    # Merge InSAR x ORTHO
    merged_v = pd.merge(
        agg_cell[['date', 'dV']], 
        ortho_v_cell[['date', 'disp']], 
        on='date', how='inner'
    ).rename(columns={'dV':'disp_insar', 'disp':'disp_ortho'})
    
    merged_h = pd.merge(
        agg_cell[['date', 'dH']], 
        ortho_h_cell[['date', 'disp']], 
        on='date', how='inner'
    ).rename(columns={'dH':'disp_insar', 'disp':'disp_ortho'})

    # Scatter plot dV
    if not merged_v.empty:
        ax.scatter(merged_v['disp_ortho'], merged_v['disp_insar'], color='blue', alpha=0.6, label='dV')
        rmse_v = np.sqrt(mean_squared_error(merged_v['disp_ortho'], merged_v['disp_insar']))
        mae_v = mean_absolute_error(merged_v['disp_ortho'], merged_v['disp_insar'])
        r_v = np.corrcoef(merged_v['disp_ortho'], merged_v['disp_insar'])[0,1]
        ax.text(0.03, 0.95, f"dV:\nr={r_v:.2f}\nRMSE={rmse_v:.2f}\nMAE={mae_v:.2f}", 
                transform=ax.transAxes, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.7), fontsize=8)

    # Scatter plot dH
    if not merged_h.empty:
        ax.scatter(merged_h['disp_ortho'], merged_h['disp_insar'], color='green', alpha=0.6, label='dH')
        rmse_h = np.sqrt(mean_squared_error(merged_h['disp_ortho'], merged_h['disp_insar']))
        mae_h = mean_absolute_error(merged_h['disp_ortho'], merged_h['disp_insar'])
        r_h = np.corrcoef(merged_h['disp_ortho'], merged_h['disp_insar'])[0,1]
        ax.text(0.03, 0.68, f"dH:\nr={r_h:.2f}\nRMSE={rmse_h:.2f}\nMAE={mae_h:.2f}", 
                transform=ax.transAxes, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.7), fontsize=8)

    # Linha identidade
    vmin = min(merged_v['disp_ortho'].min() if not merged_v.empty else np.nan,
               merged_h['disp_ortho'].min() if not merged_h.empty else np.nan)
    vmax = max(merged_v['disp_ortho'].max() if not merged_v.empty else np.nan,
               merged_h['disp_ortho'].max() if not merged_h.empty else np.nan)
    if not np.isnan(vmin) and not np.isnan(vmax):
        ax.plot([vmin, vmax], [vmin, vmax], 'k--', lw=1)

    # Configurações do subplot
    ax.set_title(f"Célula {cid}", fontsize=10)
    ax.set_xlabel("ORTHO (mm)")
    ax.set_ylabel("InSAR (mm)")
    ax.legend(fontsize=8, loc='best')
    ax.grid(False)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)

# Remove eixos extras
for j in range(i+1, n_rows*n_cols):
    r, c = j // n_cols, j % n_cols
    fig.delaxes(axes[r,c])

plt.tight_layout()
plt.show()



# =======================================
# 13. Figura 4: Scatter global com métricas
# =======================================
merged_v_list = []
merged_h_list = []
for cid in selected_ids_plot:
    agg_cell = agg_sel[agg_sel['cell_id']==cid]
    ortho_v_cell = ortho_v_sel[ortho_v_sel['cell_id']==cid]
    ortho_h_cell = ortho_h_sel[ortho_h_sel['cell_id']==cid]
    merged_v_list.append(pd.merge(agg_cell[['cell_id','date','dV']], ortho_v_cell[['cell_id','date','disp']], on=['cell_id','date'], how='inner').rename(columns={'dV':'disp_insar','disp':'disp_ortho'}))
    merged_h_list.append(pd.merge(agg_cell[['cell_id','date','dH']], ortho_h_cell[['cell_id','date','disp']], on=['cell_id','date'], how='inner').rename(columns={'dH':'disp_insar','disp':'disp_ortho'}))

merged_v_global = pd.concat(merged_v_list, ignore_index=True)
merged_h_global = pd.concat(merged_h_list, ignore_index=True)

def compute_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    r_corr = np.corrcoef(y_true, y_pred)[0,1]
    return rmse, mae, r2, r_corr



rmse_v, mae_v, r2_v, r_v = compute_metrics(merged_v_global['disp_ortho'], merged_v_global['disp_insar'])
rmse_h, mae_h, r2_h, r_h = compute_metrics(merged_h_global['disp_ortho'], merged_h_global['disp_insar'])

fig, axes = plt.subplots(1,2,figsize=(14,6))
axes[0].scatter(merged_v_global['disp_ortho'], merged_v_global['disp_insar'], color='blue', alpha=0.3)
axes[0].plot([merged_v_global['disp_ortho'].min(), merged_v_global['disp_ortho'].max()],
             [merged_v_global['disp_ortho'].min(), merged_v_global['disp_ortho'].max()],'k--')
axes[0].set_xlabel("ORTHO dV (mm)")
axes[0].set_ylabel("InSAR dV (mm)")
axes[0].set_title("Comparação dV InSAR vs ORTHO")
axes[0].text(0.05,0.95,f"r={r_v:.2f}\nRMSE={rmse_v:.2f}\nMAE={mae_v:.2f}\nR²={r2_v:.2f}", transform=axes[0].transAxes,
             verticalalignment='top', bbox=dict(facecolor='white', alpha=0.8))

axes[1].scatter(merged_h_global['disp_ortho'], merged_h_global['disp_insar'], color='green', alpha=0.3)
axes[1].plot([merged_h_global['disp_ortho'].min(), merged_h_global['disp_ortho'].max()],
             [merged_h_global['disp_ortho'].min(), merged_h_global['disp_ortho'].max()],'k--')
axes[1].set_xlabel("ORTHO dH (mm)")
axes[1].set_ylabel("InSAR dH (mm)")
axes[1].set_title("Comparação dH InSAR vs ORTHO")
axes[1].text(0.05,0.95,f"r={r_h:.2f}\nRMSE={rmse_h:.2f}\nMAE={mae_h:.2f}\nR²={r2_h:.2f}", transform=axes[1].transAxes,
             verticalalignment='top', bbox=dict(facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

In [ ]:
# =======================================
# Script completo: Processamento e Visualizações
# =======================================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import seaborn as sns

# =======================================
# 1. Ler CSVs ASC, DESC e ORTHO
# =======================================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# =======================================
# 2. Filtrar área de interesse
# =======================================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# =======================================
# 3. Melt para ASC/DESC
# =======================================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
                      value_vars=disp_cols, var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(start=max(asc_long['date'].min(), desc_long['date'].min()),
                             end=min(asc_long['date'].max(), desc_long['date'].max()),
                             freq='MS')

# =======================================
# 4. Interpolação temporal linear
# =======================================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64),
                           group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates, 'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# =======================================
# 5. Interpolação espacial IDW
# =======================================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# =======================================
# 6. Calcular β e γ
# =======================================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# =======================================
# 7. Calcular dV e dH
# =======================================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) + np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) + dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# =======================================
# 8. Grelha centrada ORTHO
# =======================================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

agg = asc_interp.dropna(subset=['cell_x','cell_y']).groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# =======================================
# 9. Selecionar células
# =======================================
selected_ids_plot = ["3_5","4_5","5_5","6_5","7_5","3_4","4_4","5_4","6_4","5_6","6_6","7_6"]
agg_sel = agg[agg['cell_id'].isin(selected_ids_plot)]

def melt_ortho(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing'], value_vars=disp_cols,
                      var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

ortho_v_long = melt_ortho(ortho_v)
ortho_h_long = melt_ortho(ortho_h)

for ortho_df in [ortho_v_long, ortho_h_long]:
    ortho_df['cell_x'] = pd.cut(ortho_df['easting'], bins=x_edges_shifted, labels=False)
    ortho_df['cell_y'] = pd.cut(ortho_df['northing'], bins=y_edges_shifted, labels=False)
    ortho_df.dropna(subset=['cell_x','cell_y'], inplace=True)
    ortho_df['cell_id'] = ortho_df['cell_x'].astype(int).astype(str) + "_" + ortho_df['cell_y'].astype(int).astype(str)

ortho_v_sel = ortho_v_long[ortho_v_long['cell_id'].isin(selected_ids_plot)]
ortho_h_sel = ortho_h_long[ortho_h_long['cell_id'].isin(selected_ids_plot)]

# =======================================
# 10. Figura 1: Mapa das células e pontos
# =======================================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({"cell_x": ix, "cell_y": iy, "cell_id": f"{ix}_{iy}",
                          "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                                          x_edges_shifted[ix+1], y_edges_shifted[iy+1])})
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)
points = agg.groupby('cell_id').first().reset_index()
gdf_points = gpd.GeoDataFrame(points, geometry=gpd.points_from_xy(points['x_center'], points['y_center']),
                              crs="EPSG:3035").to_crs(epsg=3857)
grid_sel = grid[grid['cell_id'].isin(selected_ids_plot)]
points_sel = gdf_points[gdf_points['cell_id'].isin(selected_ids_plot)]

fig_map, ax_map = plt.subplots(figsize=(10,10))
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5, label=f'Grelha Base ({grid_size} m)')
grid_sel.boundary.plot(ax=ax_map, color='red', linewidth=1.5, label=f'Células Selecionadas ({grid_size} m)')
points_sel.plot(ax=ax_map, color='white', edgecolor='black', markersize=60, label='Pontos ASC/DESC')

ortho_comb = pd.concat([ortho_v_sel, ortho_h_sel], ignore_index=True).drop_duplicates(subset=['easting','northing','date'])
gdf_ortho_comb = gpd.GeoDataFrame(ortho_comb, geometry=gpd.points_from_xy(ortho_comb['easting'], ortho_comb['northing']),
                                  crs="EPSG:3035").to_crs(epsg=3857)
if not gdf_ortho_comb.empty:
    gdf_ortho_comb.plot(ax=ax_map, color='black', markersize=25, alpha=0.8, label='Pontos ORTHO')

for _, row in grid_sel.iterrows():
    x_min, y_min, x_max, y_max = row['geometry'].bounds
    ax_map.text(x_min, y_max, row['cell_id'], fontsize=9, ha='left', va='top',
                bbox=dict(facecolor='white', alpha=0.6, pad=1))
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title("Mapa das Células Selecionadas", fontsize=14)
ax_map.set_axis_off()
ax_map.legend(loc='lower left', fontsize=9)
plt.show()

# =======================================
# 11. Figura 2: Séries temporais InSAR vs ORTHO
# =======================================
n_cells = len(selected_ids_plot)
n_cols = 3
n_rows = int(np.ceil(n_cells / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows*3), sharex=True, sharey=True)
if n_rows==1: axes=axes.reshape(1,-1)

v_min = min(agg_sel['dV'].min(), agg_sel['dH'].min(),
            ortho_v_sel['disp'].min(), ortho_h_sel['disp'].min())
v_max = max(agg_sel['dV'].max(), agg_sel['dH'].max(),
            ortho_v_sel['disp'].max(), ortho_h_sel['disp'].max())

for i, cid in enumerate(selected_ids_plot):
    r, c = i // n_cols, i % n_cols
    ax = axes[r, c]
    agg_cell = agg_sel[agg_sel['cell_id']==cid]
    ortho_v_cell = ortho_v_sel[ortho_v_sel['cell_id']==cid]
    ortho_h_cell = ortho_h_sel[ortho_h_sel['cell_id']==cid]

    ax.plot(agg_cell['date'], agg_cell['dV'], color='blue', label='ASC/DESC dV')
    ax.plot(agg_cell['date'], agg_cell['dH'], color='green', label='ASC/DESC dH')
    if not ortho_v_cell.empty: ax.plot(ortho_v_cell['date'], ortho_v_cell['disp'], color='blue', linestyle='--', label='ORTHO dV')
    if not ortho_h_cell.empty: ax.plot(ortho_h_cell['date'], ortho_h_cell['disp'], color='green', linestyle='--', label='ORTHO dH')
    ax.set_title(f"Célula {cid}", fontsize=10)
    ax.set_xlabel("Data")
    ax.set_ylabel("Deslocamento (mm)")
    ax.set_ylim(v_min,v_max)
    ax.legend(fontsize=8, loc='best')

for j in range(i+1, n_rows*n_cols):
    r, c = j // n_cols, j % n_cols
    fig.delaxes(axes[r,c])
plt.tight_layout()
plt.show()

# =======================================
# 12. Figura 3: Scatter InSAR vs ORTHO por célula (com métricas identificadas)
# =======================================
n_cols = 3
n_rows = int(np.ceil(n_cells / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows*3), sharex=True, sharey=True)
if n_rows == 1: 
    axes = axes.reshape(1, -1)

for i, cid in enumerate(selected_ids_plot):
    r, c = i // n_cols, i % n_cols
    ax = axes[r, c]

    # Dados da célula
    agg_cell = agg_sel[agg_sel['cell_id'] == cid]
    ortho_v_cell = ortho_v_sel[ortho_v_sel['cell_id'] == cid]
    ortho_h_cell = ortho_h_sel[ortho_h_sel['cell_id'] == cid]

    # Merge InSAR x ORTHO
    merged_v = pd.merge(
        agg_cell[['date', 'dV']], 
        ortho_v_cell[['date', 'disp']], 
        on='date', how='inner'
    ).rename(columns={'dV':'disp_insar', 'disp':'disp_ortho'})
    
    merged_h = pd.merge(
        agg_cell[['date', 'dH']], 
        ortho_h_cell[['date', 'disp']], 
        on='date', how='inner'
    ).rename(columns={'dH':'disp_insar', 'disp':'disp_ortho'})

    # Scatter plot dV
    if not merged_v.empty:
        ax.scatter(merged_v['disp_ortho'], merged_v['disp_insar'], color='blue', alpha=0.6, label='dV')
        rmse_v = np.sqrt(mean_squared_error(merged_v['disp_ortho'], merged_v['disp_insar']))
        mae_v = mean_absolute_error(merged_v['disp_ortho'], merged_v['disp_insar'])
        r_v = np.corrcoef(merged_v['disp_ortho'], merged_v['disp_insar'])[0,1]
        ax.text(0.03, 0.95, f"dV:\nr={r_v:.2f}\nRMSE={rmse_v:.2f}\nMAE={mae_v:.2f}", 
                transform=ax.transAxes, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.7), fontsize=9)

    # Scatter plot dH
    if not merged_h.empty:
        ax.scatter(merged_h['disp_ortho'], merged_h['disp_insar'], color='green', alpha=0.6, label='dH')
        rmse_h = np.sqrt(mean_squared_error(merged_h['disp_ortho'], merged_h['disp_insar']))
        mae_h = mean_absolute_error(merged_h['disp_ortho'], merged_h['disp_insar'])
        r_h = np.corrcoef(merged_h['disp_ortho'], merged_h['disp_insar'])[0,1]
        ax.text(0.03, 0.62, f"dH:\nr={r_h:.2f}\nRMSE={rmse_h:.2f}\nMAE={mae_h:.2f}", 
                transform=ax.transAxes, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.7), fontsize=9)

    # Linha identidade
    vmin = min(merged_v['disp_ortho'].min() if not merged_v.empty else np.nan,
               merged_h['disp_ortho'].min() if not merged_h.empty else np.nan)
    vmax = max(merged_v['disp_ortho'].max() if not merged_v.empty else np.nan,
               merged_h['disp_ortho'].max() if not merged_h.empty else np.nan)
    if not np.isnan(vmin) and not np.isnan(vmax):
        ax.plot([vmin, vmax], [vmin, vmax], 'k--', lw=1)

    # Configurações do subplot
    ax.set_title(f"Célula {cid}", fontsize=10)
    ax.set_xlabel("ORTHO (mm)")
    ax.set_ylabel("InSAR (mm)")
    ax.legend(fontsize=9, loc='best')
    ax.grid(False)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)

# Remove eixos extras
for j in range(i+1, n_rows*n_cols):
    r, c = j // n_cols, j % n_cols
    fig.delaxes(axes[r,c])

plt.tight_layout()
plt.show()



# =======================================
# 13. Figura 4: Scatter global com métricas (sem R²)
# =======================================
merged_v_list = []
merged_h_list = []
for cid in selected_ids_plot:
    agg_cell = agg_sel[agg_sel['cell_id']==cid]
    ortho_v_cell = ortho_v_sel[ortho_v_sel['cell_id']==cid]
    ortho_h_cell = ortho_h_sel[ortho_h_sel['cell_id']==cid]
    merged_v_list.append(pd.merge(
        agg_cell[['cell_id','date','dV']], ortho_v_cell[['cell_id','date','disp']],
        on=['cell_id','date'], how='inner'
    ).rename(columns={'dV':'disp_insar','disp':'disp_ortho'}))
    merged_h_list.append(pd.merge(
        agg_cell[['cell_id','date','dH']], ortho_h_cell[['cell_id','date','disp']],
        on=['cell_id','date'], how='inner'
    ).rename(columns={'dH':'disp_insar','disp':'disp_ortho'}))

merged_v_global = pd.concat(merged_v_list, ignore_index=True)
merged_h_global = pd.concat(merged_h_list, ignore_index=True)

def compute_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r_corr = np.corrcoef(y_true, y_pred)[0,1]
    return rmse, mae, r_corr

rmse_v, mae_v, r_v = compute_metrics(merged_v_global['disp_ortho'], merged_v_global['disp_insar'])
rmse_h, mae_h, r_h = compute_metrics(merged_h_global['disp_ortho'], merged_h_global['disp_insar'])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ---- dV ----
axes[0].scatter(merged_v_global['disp_ortho'], merged_v_global['disp_insar'], color='blue', alpha=0.3)
axes[0].plot([merged_v_global['disp_ortho'].min(), merged_v_global['disp_ortho'].max()],
             [merged_v_global['disp_ortho'].min(), merged_v_global['disp_ortho'].max()], 'k--')
axes[0].set_xlabel("ORTHO dV (mm)", fontsize=12)
axes[0].set_ylabel("InSAR dV (mm)", fontsize=12)
axes[0].set_title("Comparação dV InSAR vs ORTHO", fontsize=13)
axes[0].text(0.05, 0.95, f"r={r_v:.2f}\nRMSE={rmse_v:.2f}\nMAE={mae_v:.2f}",
             transform=axes[0].transAxes, verticalalignment='top',
             bbox=dict(facecolor='white', alpha=0.8), fontsize=11)

# ---- dH ----
axes[1].scatter(merged_h_global['disp_ortho'], merged_h_global['disp_insar'], color='green', alpha=0.3)
axes[1].plot([merged_h_global['disp_ortho'].min(), merged_h_global['disp_ortho'].max()],
             [merged_h_global['disp_ortho'].min(), merged_h_global['disp_ortho'].max()], 'k--')
axes[1].set_xlabel("ORTHO dH (mm)", fontsize=12)
axes[1].set_ylabel("InSAR dH (mm)", fontsize=12)
axes[1].set_title("Comparação dH InSAR vs ORTHO", fontsize=13)
axes[1].text(0.05, 0.95, f"r={r_h:.2f}\nRMSE={rmse_h:.2f}\nMAE={mae_h:.2f}",
             transform=axes[1].transAxes, verticalalignment='top',
             bbox=dict(facecolor='white', alpha=0.8), fontsize=11)

plt.tight_layout()
plt.show()


# Artigo 1

para todas as células e com
k = 5
r = 25
p = 2

In [ ]:
# =======================================
# Script completo: Processamento e Visualizações
# =======================================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import seaborn as sns

# =======================================
# 1. Ler CSVs ASC, DESC e ORTHO
# =======================================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# =======================================
# 2. Filtrar área de interesse
# =======================================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# =======================================
# 3. Melt para ASC/DESC
# =======================================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
                      value_vars=disp_cols, var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(start=max(asc_long['date'].min(), desc_long['date'].min()),
                             end=min(asc_long['date'].max(), desc_long['date'].max()),
                             freq='MS')

# =======================================
# 4. Interpolação temporal linear
# =======================================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64),
                           group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates, 'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# =======================================
# 5. Interpolação espacial IDW
# =======================================
def idw_interpolation_per_date(source_df, target_df, radius=25, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# =======================================
# 6. Calcular β e γ
# =======================================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# =======================================
# 7. Calcular dV e dH
# =======================================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) + np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) + dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# =======================================
# 8. Grelha centrada ORTHO
# =======================================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

agg = asc_interp.dropna(subset=['cell_x','cell_y']).groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# =======================================
# 9. Selecionar células
# =======================================
selected_ids_plot = ["3_5","4_5","5_5","6_5","7_5","3_4","4_4","5_4","6_4","5_6","6_6","7_6"]
agg_sel = agg[agg['cell_id'].isin(selected_ids_plot)]

def melt_ortho(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing'], value_vars=disp_cols,
                      var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

ortho_v_long = melt_ortho(ortho_v)
ortho_h_long = melt_ortho(ortho_h)

for ortho_df in [ortho_v_long, ortho_h_long]:
    ortho_df['cell_x'] = pd.cut(ortho_df['easting'], bins=x_edges_shifted, labels=False)
    ortho_df['cell_y'] = pd.cut(ortho_df['northing'], bins=y_edges_shifted, labels=False)
    ortho_df.dropna(subset=['cell_x','cell_y'], inplace=True)
    ortho_df['cell_id'] = ortho_df['cell_x'].astype(int).astype(str) + "_" + ortho_df['cell_y'].astype(int).astype(str)

ortho_v_sel = ortho_v_long[ortho_v_long['cell_id'].isin(selected_ids_plot)]
ortho_h_sel = ortho_h_long[ortho_h_long['cell_id'].isin(selected_ids_plot)]

# =======================================
# 10. Figura 1: Mapa das células e pontos
# =======================================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({"cell_x": ix, "cell_y": iy, "cell_id": f"{ix}_{iy}",
                          "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                                          x_edges_shifted[ix+1], y_edges_shifted[iy+1])})
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)
points = agg.groupby('cell_id').first().reset_index()
gdf_points = gpd.GeoDataFrame(points, geometry=gpd.points_from_xy(points['x_center'], points['y_center']),
                              crs="EPSG:3035").to_crs(epsg=3857)
grid_sel = grid[grid['cell_id'].isin(selected_ids_plot)]
points_sel = gdf_points[gdf_points['cell_id'].isin(selected_ids_plot)]

fig_map, ax_map = plt.subplots(figsize=(10,10))
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5, label=f'Grelha Base ({grid_size} m)')
grid_sel.boundary.plot(ax=ax_map, color='red', linewidth=1.5, label=f'Células Selecionadas ({grid_size} m)')
points_sel.plot(ax=ax_map, color='white', edgecolor='black', markersize=60, label='Pontos ASC/DESC')

ortho_comb = pd.concat([ortho_v_sel, ortho_h_sel], ignore_index=True).drop_duplicates(subset=['easting','northing','date'])
gdf_ortho_comb = gpd.GeoDataFrame(ortho_comb, geometry=gpd.points_from_xy(ortho_comb['easting'], ortho_comb['northing']),
                                  crs="EPSG:3035").to_crs(epsg=3857)
if not gdf_ortho_comb.empty:
    gdf_ortho_comb.plot(ax=ax_map, color='black', markersize=25, alpha=0.8, label='Pontos ORTHO')

for _, row in grid_sel.iterrows():
    x_min, y_min, x_max, y_max = row['geometry'].bounds
    ax_map.text(x_min, y_max, row['cell_id'], fontsize=9, ha='left', va='top',
                bbox=dict(facecolor='white', alpha=0.6, pad=1))
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title("Mapa das Células Selecionadas", fontsize=14)
ax_map.set_axis_off()
ax_map.legend(loc='lower left', fontsize=9)
plt.show()

# =======================================
# 11. Figura 2: Séries temporais InSAR vs ORTHO
# =======================================
n_cells = len(selected_ids_plot)
n_cols = 3
n_rows = int(np.ceil(n_cells / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows*3), sharex=True, sharey=True)
if n_rows==1: axes=axes.reshape(1,-1)

v_min = min(agg_sel['dV'].min(), agg_sel['dH'].min(),
            ortho_v_sel['disp'].min(), ortho_h_sel['disp'].min())
v_max = max(agg_sel['dV'].max(), agg_sel['dH'].max(),
            ortho_v_sel['disp'].max(), ortho_h_sel['disp'].max())

for i, cid in enumerate(selected_ids_plot):
    r, c = i // n_cols, i % n_cols
    ax = axes[r, c]
    agg_cell = agg_sel[agg_sel['cell_id']==cid]
    ortho_v_cell = ortho_v_sel[ortho_v_sel['cell_id']==cid]
    ortho_h_cell = ortho_h_sel[ortho_h_sel['cell_id']==cid]

    ax.plot(agg_cell['date'], agg_cell['dV'], color='blue', label='ASC/DESC dV')
    ax.plot(agg_cell['date'], agg_cell['dH'], color='green', label='ASC/DESC dH')
    if not ortho_v_cell.empty: ax.plot(ortho_v_cell['date'], ortho_v_cell['disp'], color='blue', linestyle='--', label='ORTHO dV')
    if not ortho_h_cell.empty: ax.plot(ortho_h_cell['date'], ortho_h_cell['disp'], color='green', linestyle='--', label='ORTHO dH')
    ax.set_title(f"Célula {cid}", fontsize=10)
    ax.set_xlabel("Data")
    ax.set_ylabel("Deslocamento (mm)")
    ax.set_ylim(v_min,v_max)
    ax.legend(fontsize=8, loc='best')

for j in range(i+1, n_rows*n_cols):
    r, c = j // n_cols, j % n_cols
    fig.delaxes(axes[r,c])
plt.tight_layout()
plt.show()

# =======================================
# 12. Figura 3: Scatter InSAR vs ORTHO por célula (com métricas identificadas)
# =======================================
n_cols = 3
n_rows = int(np.ceil(n_cells / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows*3), sharex=True, sharey=True)
if n_rows == 1: 
    axes = axes.reshape(1, -1)

for i, cid in enumerate(selected_ids_plot):
    r, c = i // n_cols, i % n_cols
    ax = axes[r, c]

    # Dados da célula
    agg_cell = agg_sel[agg_sel['cell_id'] == cid]
    ortho_v_cell = ortho_v_sel[ortho_v_sel['cell_id'] == cid]
    ortho_h_cell = ortho_h_sel[ortho_h_sel['cell_id'] == cid]

    # Merge InSAR x ORTHO
    merged_v = pd.merge(
        agg_cell[['date', 'dV']], 
        ortho_v_cell[['date', 'disp']], 
        on='date', how='inner'
    ).rename(columns={'dV':'disp_insar', 'disp':'disp_ortho'})
    
    merged_h = pd.merge(
        agg_cell[['date', 'dH']], 
        ortho_h_cell[['date', 'disp']], 
        on='date', how='inner'
    ).rename(columns={'dH':'disp_insar', 'disp':'disp_ortho'})

    # Scatter plot dV
    if not merged_v.empty:
        ax.scatter(merged_v['disp_ortho'], merged_v['disp_insar'], color='blue', alpha=0.6, label='dV')
        rmse_v = np.sqrt(mean_squared_error(merged_v['disp_ortho'], merged_v['disp_insar']))
        mae_v = mean_absolute_error(merged_v['disp_ortho'], merged_v['disp_insar'])
        r_v = np.corrcoef(merged_v['disp_ortho'], merged_v['disp_insar'])[0,1]
        ax.text(0.03, 0.95, f"dV:\nr={r_v:.2f}\nRMSE={rmse_v:.2f}\nMAE={mae_v:.2f}", 
                transform=ax.transAxes, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.7), fontsize=9)

    # Scatter plot dH
    if not merged_h.empty:
        ax.scatter(merged_h['disp_ortho'], merged_h['disp_insar'], color='green', alpha=0.6, label='dH')
        rmse_h = np.sqrt(mean_squared_error(merged_h['disp_ortho'], merged_h['disp_insar']))
        mae_h = mean_absolute_error(merged_h['disp_ortho'], merged_h['disp_insar'])
        r_h = np.corrcoef(merged_h['disp_ortho'], merged_h['disp_insar'])[0,1]
        ax.text(0.03, 0.62, f"dH:\nr={r_h:.2f}\nRMSE={rmse_h:.2f}\nMAE={mae_h:.2f}", 
                transform=ax.transAxes, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.7), fontsize=9)

    # Linha identidade
    vmin = min(merged_v['disp_ortho'].min() if not merged_v.empty else np.nan,
               merged_h['disp_ortho'].min() if not merged_h.empty else np.nan)
    vmax = max(merged_v['disp_ortho'].max() if not merged_v.empty else np.nan,
               merged_h['disp_ortho'].max() if not merged_h.empty else np.nan)
    if not np.isnan(vmin) and not np.isnan(vmax):
        ax.plot([vmin, vmax], [vmin, vmax], 'k--', lw=1)

    # Configurações do subplot
    ax.set_title(f"Célula {cid}", fontsize=10)
    ax.set_xlabel("ORTHO (mm)")
    ax.set_ylabel("InSAR (mm)")
    ax.legend(fontsize=9, loc='best')
    ax.grid(False)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)

# Remove eixos extras
for j in range(i+1, n_rows*n_cols):
    r, c = j // n_cols, j % n_cols
    fig.delaxes(axes[r,c])

plt.tight_layout()
plt.show()



# =======================================
# 13. Figura 4: Scatter global com métricas (sem R²)
# =======================================
merged_v_list = []
merged_h_list = []
for cid in selected_ids_plot:
    agg_cell = agg_sel[agg_sel['cell_id']==cid]
    ortho_v_cell = ortho_v_sel[ortho_v_sel['cell_id']==cid]
    ortho_h_cell = ortho_h_sel[ortho_h_sel['cell_id']==cid]
    merged_v_list.append(pd.merge(
        agg_cell[['cell_id','date','dV']], ortho_v_cell[['cell_id','date','disp']],
        on=['cell_id','date'], how='inner'
    ).rename(columns={'dV':'disp_insar','disp':'disp_ortho'}))
    merged_h_list.append(pd.merge(
        agg_cell[['cell_id','date','dH']], ortho_h_cell[['cell_id','date','disp']],
        on=['cell_id','date'], how='inner'
    ).rename(columns={'dH':'disp_insar','disp':'disp_ortho'}))

merged_v_global = pd.concat(merged_v_list, ignore_index=True)
merged_h_global = pd.concat(merged_h_list, ignore_index=True)

def compute_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r_corr = np.corrcoef(y_true, y_pred)[0,1]
    return rmse, mae, r_corr

rmse_v, mae_v, r_v = compute_metrics(merged_v_global['disp_ortho'], merged_v_global['disp_insar'])
rmse_h, mae_h, r_h = compute_metrics(merged_h_global['disp_ortho'], merged_h_global['disp_insar'])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ---- dV ----
axes[0].scatter(merged_v_global['disp_ortho'], merged_v_global['disp_insar'], color='blue', alpha=0.3)
axes[0].plot([merged_v_global['disp_ortho'].min(), merged_v_global['disp_ortho'].max()],
             [merged_v_global['disp_ortho'].min(), merged_v_global['disp_ortho'].max()], 'k--')
axes[0].set_xlabel("ORTHO dV (mm)", fontsize=12)
axes[0].set_ylabel("InSAR dV (mm)", fontsize=12)
axes[0].set_title("Comparação dV InSAR vs ORTHO", fontsize=13)
axes[0].text(0.05, 0.95, f"r={r_v:.2f}\nRMSE={rmse_v:.2f}\nMAE={mae_v:.2f}",
             transform=axes[0].transAxes, verticalalignment='top',
             bbox=dict(facecolor='white', alpha=0.8), fontsize=11)

# ---- dH ----
axes[1].scatter(merged_h_global['disp_ortho'], merged_h_global['disp_insar'], color='green', alpha=0.3)
axes[1].plot([merged_h_global['disp_ortho'].min(), merged_h_global['disp_ortho'].max()],
             [merged_h_global['disp_ortho'].min(), merged_h_global['disp_ortho'].max()], 'k--')
axes[1].set_xlabel("ORTHO dH (mm)", fontsize=12)
axes[1].set_ylabel("InSAR dH (mm)", fontsize=12)
axes[1].set_title("Comparação dH InSAR vs ORTHO", fontsize=13)
axes[1].text(0.05, 0.95, f"r={r_h:.2f}\nRMSE={rmse_h:.2f}\nMAE={mae_h:.2f}",
             transform=axes[1].transAxes, verticalalignment='top',
             bbox=dict(facecolor='white', alpha=0.8), fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
# =======================================
# Script completo: Processamento e Visualizações
# =======================================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import seaborn as sns

# =======================================
# 1. Ler CSVs ASC, DESC e ORTHO
# =======================================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# =======================================
# 2. Filtrar área de interesse
# =======================================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# =======================================
# 3. Melt para ASC/DESC
# =======================================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
                      value_vars=disp_cols, var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(start=max(asc_long['date'].min(), desc_long['date'].min()),
                             end=min(asc_long['date'].max(), desc_long['date'].max()),
                             freq='MS')

# =======================================
# 4. Interpolação temporal linear
# =======================================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64),
                           group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates, 'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# =======================================
# 5. Interpolação espacial IDW
# =======================================
def idw_interpolation_per_date(source_df, target_df, radius=25, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# =======================================
# 6. Calcular β e γ
# =======================================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# =======================================
# 7. Calcular dV e dH
# =======================================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) + np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) + dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# =======================================
# 8. Grelha centrada ORTHO
# =======================================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

agg = asc_interp.dropna(subset=['cell_x','cell_y']).groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# =======================================
# 9. Selecionar amostra aleatória de 9 células
# =======================================
# 1. Pegamos em todos os IDs únicos que existem no dataframe agg
all_available_ids = agg['cell_id'].unique()

# 2. Escolhemos 9 aleatoriamente (random_state garante que sejam sempre as mesmas 
# cada vez que corres o código, se quiseres mudar, remove o random_state)
import random
selected_ids_plot = pd.Series(all_available_ids).sample(n=6, random_state=15).tolist()

# O resto do código permanece igual
agg_sel = agg[agg['cell_id'].isin(selected_ids_plot)]

def melt_ortho(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing'], value_vars=disp_cols,
                      var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

ortho_v_long = melt_ortho(ortho_v)
ortho_h_long = melt_ortho(ortho_h)

for ortho_df in [ortho_v_long, ortho_h_long]:
    ortho_df['cell_x'] = pd.cut(ortho_df['easting'], bins=x_edges_shifted, labels=False)
    ortho_df['cell_y'] = pd.cut(ortho_df['northing'], bins=y_edges_shifted, labels=False)
    ortho_df.dropna(subset=['cell_x','cell_y'], inplace=True)
    ortho_df['cell_id'] = ortho_df['cell_x'].astype(int).astype(str) + "_" + ortho_df['cell_y'].astype(int).astype(str)

ortho_v_sel = ortho_v_long[ortho_v_long['cell_id'].isin(selected_ids_plot)]
ortho_h_sel = ortho_h_long[ortho_h_long['cell_id'].isin(selected_ids_plot)]

# =======================================
# 10. Figura 1: Mapa das células e pontos
# =======================================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({"cell_x": ix, "cell_y": iy, "cell_id": f"{ix}_{iy}",
                          "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                                          x_edges_shifted[ix+1], y_edges_shifted[iy+1])})
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)
points = agg.groupby('cell_id').first().reset_index()
gdf_points = gpd.GeoDataFrame(points, geometry=gpd.points_from_xy(points['x_center'], points['y_center']),
                              crs="EPSG:3035").to_crs(epsg=3857)
grid_sel = grid[grid['cell_id'].isin(selected_ids_plot)]
points_sel = gdf_points[gdf_points['cell_id'].isin(selected_ids_plot)]

fig_map, ax_map = plt.subplots(figsize=(10,10))
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5, label=f'Grelha Base ({grid_size} m)')
grid_sel.boundary.plot(ax=ax_map, color='red', linewidth=1.5, label=f'Células Selecionadas ({grid_size} m)')
points_sel.plot(ax=ax_map, color='white', edgecolor='black', markersize=60, label='Pontos ASC/DESC')

ortho_comb = pd.concat([ortho_v_sel, ortho_h_sel], ignore_index=True).drop_duplicates(subset=['easting','northing','date'])
gdf_ortho_comb = gpd.GeoDataFrame(ortho_comb, geometry=gpd.points_from_xy(ortho_comb['easting'], ortho_comb['northing']),
                                  crs="EPSG:3035").to_crs(epsg=3857)
if not gdf_ortho_comb.empty:
    gdf_ortho_comb.plot(ax=ax_map, color='black', markersize=25, alpha=0.8, label='Pontos ORTHO')

for _, row in grid_sel.iterrows():
    x_min, y_min, x_max, y_max = row['geometry'].bounds
    ax_map.text(x_min, y_max, row['cell_id'], fontsize=9, ha='left', va='top',
                bbox=dict(facecolor='white', alpha=0.6, pad=1))
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title("Mapa das Células Selecionadas", fontsize=14)
ax_map.set_axis_off()
ax_map.legend(loc='lower left', fontsize=9)

# plt.savefig("random_cells.pdf", format='pdf')
plt.savefig("figuras/random_cells.pdf", bbox_inches='tight', pad_inches=0.02)

plt.show()

# =======================================
# 11. Figura 2: Séries temporais InSAR vs ORTHO
# =======================================
n_cells = len(selected_ids_plot)
n_cols = 3
n_rows = int(np.ceil(n_cells / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows*3), sharex=True, sharey=True)
if n_rows==1: axes=axes.reshape(1,-1)

v_min = min(agg_sel['dV'].min(), agg_sel['dH'].min(),
            ortho_v_sel['disp'].min(), ortho_h_sel['disp'].min())
v_max = max(agg_sel['dV'].max(), agg_sel['dH'].max(),
            ortho_v_sel['disp'].max(), ortho_h_sel['disp'].max())

for i, cid in enumerate(selected_ids_plot):
    r, c = i // n_cols, i % n_cols
    ax = axes[r, c]
    agg_cell = agg_sel[agg_sel['cell_id']==cid]
    ortho_v_cell = ortho_v_sel[ortho_v_sel['cell_id']==cid]
    ortho_h_cell = ortho_h_sel[ortho_h_sel['cell_id']==cid]

    ax.plot(agg_cell['date'], agg_cell['dV'], color='blue', label='ASC/DESC dV')
    ax.plot(agg_cell['date'], agg_cell['dH'], color='green', label='ASC/DESC dH')
    if not ortho_v_cell.empty: ax.plot(ortho_v_cell['date'], ortho_v_cell['disp'], color='blue', linestyle='--', label='ORTHO dV')
    if not ortho_h_cell.empty: ax.plot(ortho_h_cell['date'], ortho_h_cell['disp'], color='green', linestyle='--', label='ORTHO dH')
    ax.set_title(f"Célula {cid}", fontsize=10)
    ax.set_xlabel("Data")
    ax.set_ylabel("Deslocamento (mm)")
    ax.set_ylim(v_min,v_max)
    ax.legend(fontsize=8, loc='best')

for j in range(i+1, n_rows*n_cols):
    r, c = j // n_cols, j % n_cols
    fig.delaxes(axes[r,c])
plt.tight_layout()

# plt.savefig("random_disp.pdf", format='pdf')
plt.savefig("figuras/random_disp.pdf", bbox_inches='tight', pad_inches=0.02)

plt.show()

# =======================================
# 12. Figura 3: Scatter InSAR vs ORTHO por célula (com métricas identificadas)
# =======================================
n_cols = 3
n_rows = int(np.ceil(n_cells / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows*3), sharex=True, sharey=True)
if n_rows == 1: 
    axes = axes.reshape(1, -1)

for i, cid in enumerate(selected_ids_plot):
    r, c = i // n_cols, i % n_cols
    ax = axes[r, c]

    # Dados da célula
    agg_cell = agg_sel[agg_sel['cell_id'] == cid]
    ortho_v_cell = ortho_v_sel[ortho_v_sel['cell_id'] == cid]
    ortho_h_cell = ortho_h_sel[ortho_h_sel['cell_id'] == cid]

    # Merge InSAR x ORTHO
    merged_v = pd.merge(
        agg_cell[['date', 'dV']], 
        ortho_v_cell[['date', 'disp']], 
        on='date', how='inner'
    ).rename(columns={'dV':'disp_insar', 'disp':'disp_ortho'})
    
    merged_h = pd.merge(
        agg_cell[['date', 'dH']], 
        ortho_h_cell[['date', 'disp']], 
        on='date', how='inner'
    ).rename(columns={'dH':'disp_insar', 'disp':'disp_ortho'})

    # Scatter plot dV
    if not merged_v.empty:
        ax.scatter(merged_v['disp_ortho'], merged_v['disp_insar'], color='blue', alpha=0.6, label='dV')
        rmse_v = np.sqrt(mean_squared_error(merged_v['disp_ortho'], merged_v['disp_insar']))
        mae_v = mean_absolute_error(merged_v['disp_ortho'], merged_v['disp_insar'])
        r_v = np.corrcoef(merged_v['disp_ortho'], merged_v['disp_insar'])[0,1]
        ax.text(0.03, 0.95, f"dV:\nr={r_v:.2f}\nRMSE={rmse_v:.2f}\nMAE={mae_v:.2f}", 
                transform=ax.transAxes, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.7), fontsize=9)

    # Scatter plot dH
    if not merged_h.empty:
        ax.scatter(merged_h['disp_ortho'], merged_h['disp_insar'], color='green', alpha=0.6, label='dH')
        rmse_h = np.sqrt(mean_squared_error(merged_h['disp_ortho'], merged_h['disp_insar']))
        mae_h = mean_absolute_error(merged_h['disp_ortho'], merged_h['disp_insar'])
        r_h = np.corrcoef(merged_h['disp_ortho'], merged_h['disp_insar'])[0,1]
        ax.text(0.03, 0.62, f"dH:\nr={r_h:.2f}\nRMSE={rmse_h:.2f}\nMAE={mae_h:.2f}", 
                transform=ax.transAxes, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.7), fontsize=9)

    # Linha identidade
    vmin = min(merged_v['disp_ortho'].min() if not merged_v.empty else np.nan,
               merged_h['disp_ortho'].min() if not merged_h.empty else np.nan)
    vmax = max(merged_v['disp_ortho'].max() if not merged_v.empty else np.nan,
               merged_h['disp_ortho'].max() if not merged_h.empty else np.nan)
    if not np.isnan(vmin) and not np.isnan(vmax):
        ax.plot([vmin, vmax], [vmin, vmax], 'k--', lw=1)

    # Configurações do subplot
    ax.set_title(f"Célula {cid}", fontsize=10)
    ax.set_xlabel("ORTHO (mm)")
    ax.set_ylabel("InSAR (mm)")
    ax.legend(fontsize=9, loc='best')
    ax.grid(False)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)

# Remove eixos extras
for j in range(i+1, n_rows*n_cols):
    r, c = j // n_cols, j % n_cols
    fig.delaxes(axes[r,c])

plt.tight_layout()

# plt.savefig("random_scatter.pdf", format='pdf')
plt.savefig("figuras/random_scatter.pdf", bbox_inches='tight', pad_inches=0.02)

plt.show()

# =======================================
# 13. Figura 4: Scatter global com métricas (sem R²)
# =======================================
merged_v_list = []
merged_h_list = []
for cid in selected_ids_plot:
    agg_cell = agg_sel[agg_sel['cell_id']==cid]
    ortho_v_cell = ortho_v_sel[ortho_v_sel['cell_id']==cid]
    ortho_h_cell = ortho_h_sel[ortho_h_sel['cell_id']==cid]
    merged_v_list.append(pd.merge(
        agg_cell[['cell_id','date','dV']], ortho_v_cell[['cell_id','date','disp']],
        on=['cell_id','date'], how='inner'
    ).rename(columns={'dV':'disp_insar','disp':'disp_ortho'}))
    merged_h_list.append(pd.merge(
        agg_cell[['cell_id','date','dH']], ortho_h_cell[['cell_id','date','disp']],
        on=['cell_id','date'], how='inner'
    ).rename(columns={'dH':'disp_insar','disp':'disp_ortho'}))

merged_v_global = pd.concat(merged_v_list, ignore_index=True)
merged_h_global = pd.concat(merged_h_list, ignore_index=True)

def compute_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r_corr = np.corrcoef(y_true, y_pred)[0,1]
    return rmse, mae, r_corr

rmse_v, mae_v, r_v = compute_metrics(merged_v_global['disp_ortho'], merged_v_global['disp_insar'])
rmse_h, mae_h, r_h = compute_metrics(merged_h_global['disp_ortho'], merged_h_global['disp_insar'])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ---- dV ----
axes[0].scatter(merged_v_global['disp_ortho'], merged_v_global['disp_insar'], color='blue', alpha=0.3)
axes[0].plot([merged_v_global['disp_ortho'].min(), merged_v_global['disp_ortho'].max()],
             [merged_v_global['disp_ortho'].min(), merged_v_global['disp_ortho'].max()], 'k--')
axes[0].set_xlabel("ORTHO dV (mm)", fontsize=12)
axes[0].set_ylabel("InSAR dV (mm)", fontsize=12)
axes[0].set_title("Comparação dV InSAR vs ORTHO", fontsize=13)
axes[0].text(0.05, 0.95, f"r={r_v:.2f}\nRMSE={rmse_v:.2f}\nMAE={mae_v:.2f}",
             transform=axes[0].transAxes, verticalalignment='top',
             bbox=dict(facecolor='white', alpha=0.8), fontsize=11)

# ---- dH ----
axes[1].scatter(merged_h_global['disp_ortho'], merged_h_global['disp_insar'], color='green', alpha=0.3)
axes[1].plot([merged_h_global['disp_ortho'].min(), merged_h_global['disp_ortho'].max()],
             [merged_h_global['disp_ortho'].min(), merged_h_global['disp_ortho'].max()], 'k--')
axes[1].set_xlabel("ORTHO dH (mm)", fontsize=12)
axes[1].set_ylabel("InSAR dH (mm)", fontsize=12)
axes[1].set_title("Comparação dH InSAR vs ORTHO", fontsize=13)
axes[1].text(0.05, 0.95, f"r={r_h:.2f}\nRMSE={rmse_h:.2f}\nMAE={mae_h:.2f}",
             transform=axes[1].transAxes, verticalalignment='top',
             bbox=dict(facecolor='white', alpha=0.8), fontsize=11)

plt.tight_layout()

# plt.savefig("random_global.pdf", format='pdf')
# Adiciona o parâmetro bbox_inches='tight' e pad_inches=0
plt.savefig("figuras/random_global.pdf", bbox_inches='tight', pad_inches=0.02)

plt.show()
